In [5]:
# ============================================================
# Workflow único:
# Sedighe catalog -> sim_fit reproducible -> modelos true/fit -> plot pyLIMA
# Con filtro fotométrico punto-a-punto por m5 / 5-sigma depth
# ============================================================

import os
import sys
import random
import shutil
import inspect
import importlib.util
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cycler


# ============================================================
# Configuración
# ============================================================

GLOBAL_I = 0

APPLY_PHOTOMETRIC_FILTER_FOR_FIT = True

RUNNER_PATH = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/"
    "lsstmonts_catalog_sedighe/run_lsstmonts_catalog_sedighe_xi.py"
)

CONFIG_PATH = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/"
    "lsstmonts_catalog_sedighe/config_lsstmonts_baseline_v5p3p5.json"
)

ROMAN_RUBIN_DIR = Path(
    "/home/anibal/microlensing/simulation_Rubin/roman_rubin"
)

OUT_DIR = Path(
    "/mnt/almacenamiento/sedighe_single_event_seeded_m5_filtered"
)

if str(ROMAN_RUBIN_DIR) not in sys.path:
    sys.path.insert(0, str(ROMAN_RUBIN_DIR))


# ============================================================
# Reproducibilidad
# ============================================================

@contextmanager
def deterministic_rng(seed):
    seed = int(seed)

    original_default_rng = np.random.default_rng
    master_rng = original_default_rng(seed)

    def seeded_default_rng(arg=None):
        if arg is None:
            child_seed = int(master_rng.integers(0, 2**32 - 1))
            return original_default_rng(child_seed)
        return original_default_rng(arg)

    np.random.seed(seed)
    random.seed(seed)
    np.random.default_rng = seeded_default_rng

    try:
        yield
    finally:
        np.random.default_rng = original_default_rng


def import_runner():
    """
    Importa el runner asegurando que fit_lc y functions_roman_rubin
    se lean nuevamente desde disco.
    """

    for module_name in [
        "runner_sedighe_unified",
        "fit_lc",
        "functions_roman_rubin",
    ]:
        if module_name in sys.modules:
            del sys.modules[module_name]

    sys.argv = [
        str(RUNNER_PATH),
        "--config",
        str(CONFIG_PATH),
    ]

    module_name = "runner_sedighe_unified"

    spec = importlib.util.spec_from_file_location(
        module_name,
        RUNNER_PATH,
    )

    runner = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = runner
    spec.loader.exec_module(runner)

    return runner


def force_seeded_fit(runner, seed):
    """
    Fuerza random_state=seed en fit_rubin_roman sin reemplazar run_all_fits.
    """

    import fit_lc
    import functions_roman_rubin as frr

    original_fit_rubin_roman = fit_lc.fit_rubin_roman

    def seeded_fit_rubin_roman(*args, **kwargs):
        if kwargs.get("random_state", None) is None:
            kwargs["random_state"] = int(seed)
        return original_fit_rubin_roman(*args, **kwargs)

    fit_lc.fit_rubin_roman = seeded_fit_rubin_roman
    frr.fit_rubin_roman = seeded_fit_rubin_roman

    if hasattr(frr, "run_all_fits"):
        frr.run_all_fits.__globals__["fit_rubin_roman"] = seeded_fit_rubin_roman

    if hasattr(runner, "run_all_fits"):
        runner.run_all_fits.__globals__["fit_rubin_roman"] = seeded_fit_rubin_roman


def vector_from_pylima_parameters(model_obj, pyLIMA_parameters):
    """
    Reconstruye el vector en el orden de model_dictionnary a partir de los
    parámetros YA convertidos por pyLIMA durante la simulación.

    No vuelve a convertir zero-points ni flujos.
    """

    values = []
    missing = []

    for name, _ in sorted(
        model_obj.model_dictionnary.items(),
        key=lambda x: x[1],
    ):
        try:
            value = pyLIMA_parameters[name]
        except Exception:
            if hasattr(pyLIMA_parameters, name):
                value = getattr(pyLIMA_parameters, name)
            else:
                missing.append(name)
                continue

        values.append(float(value))

    if missing:
        raise KeyError(
            "No pude reconstruir el vector true. "
            f"Faltan parámetros en pyLIMA_parameters_true: {missing}"
        )

    return np.asarray(values, dtype=float)


def count_points_by_telescope(model_obj, label):
    """
    Diagnóstico de cuántos puntos quedan después del filtrado.
    """

    print("=" * 80)
    print(label)
    print("=" * 80)

    total = 0

    for tel in model_obj.event.telescopes:
        if tel.lightcurve is None:
            n = 0
        else:
            n = len(tel.lightcurve)

        total += n
        print(f"{tel.name:5s}  n_points = {n}")

    print("-" * 80)
    print(f"TOTAL n_points = {total}")
    print("=" * 80)

    return total



# ============================================================
# Ejecutar evento
# ============================================================

runner = import_runner()

raw_catalog = runner.load_raw_catalog(
    runner.COLUMNS_FILE,
    runner.DATA_FILE,
)

prepared_catalog, invalid_catalog = runner.prepare_catalog(
    raw_catalog,
    max_base_events=GLOBAL_I + 1,
)

tasks = runner.build_tasks(
    prepared_catalog,
)

task = next(
    t for t in tasks
    if int(t["global_i"]) == int(GLOBAL_I)
)

base_row = prepared_catalog.iloc[
    int(task["prepared_index"])
].copy()


# ============================================================
# Resolver t0_jd desde el primer timestamp OpSim/MAF del campo
# ============================================================

notebook_config = {
    "path_ephemerides": str(runner.PATH_EPHEMERIDES),
    "use_roman": runner.USE_ROMAN,
    "use_rubin": runner.USE_RUBIN,
    "rubin_pointing_mode": runner.RUBIN_POINTING_MODE,
    "rubin_cache_cell_deg": runner.RUBIN_CACHE_CELL_DEG,
}

base_row = runner.apply_t0_from_first_maf_timestamp(
    base_row,
    notebook_config,
)

runner.validate_t0_first_maf_timestamp(
    base_row,
    context="notebook",
)

print("=" * 80)
print("t0 corregido desde primer timestamp OpSim/MAF")
print("=" * 80)
print(f"t0_catalog_days            = {float(base_row['t0_catalog_days']):.6f}")
print(f"t0_reference_jd            = {float(base_row['t0_reference_jd']):.6f}")
print(f"t0_jd                      = {float(base_row['t0_jd']):.6f}")
print(f"t0_jd - t0_reference_jd    = {float(base_row['t0_jd'] - base_row['t0_reference_jd']):.6f}")
print(f"t0_origin                  = {base_row['t0_origin']}")
print("=" * 80)


# ============================================================
# Catálogo local de una fila y samplers
# ============================================================

pair_catalog = runner.build_single_row_pair_catalog(
    base_row,
    task,
)

param_samplers = runner.fixed_param_samplers(
    base_row,
    task,
)

SEED = int(task["simulation_seed"])
print("SEED =", SEED)

force_seeded_fit(
    runner,
    SEED,
)


# ============================================================
# Carpetas de salida
# ============================================================

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

path_to_save_model = OUT_DIR / "models"
path_to_save_fit = OUT_DIR / "fits"
path_to_save_results = OUT_DIR / "results"

for path in [
    path_to_save_model,
    path_to_save_fit,
    path_to_save_results,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# Correr sim_fit UNA sola vez
# ============================================================

runner.install_runtime_patches()
runner.set_runtime_event_context(base_row)

try:
    kwargs = dict(
        i=SEED,

        system_type=runner.SYSTEM_TYPE,
        model=runner.MODEL,
        algo=runner.ALGO,

        path_TRILEGAL_set=None,
        path_GENULENS_set=None,

        path_to_save_model=str(path_to_save_model),
        path_to_save_fit=str(path_to_save_fit),
        path_to_save_results=str(path_to_save_results),
        path_ephemerides=str(runner.PATH_EPHEMERIDES),

        # Simulación completa.
        time_window=None,
        param_samplers=param_samplers,

        custom_system=None,
        catalog_mode="astrodatalab_pairs",
        pair_catalog=pair_catalog,
        path_pair_catalog=None,

        use_roman=runner.USE_ROMAN,
        use_rubin=runner.USE_RUBIN,

        truth_parallax=True,

        # Fit.
        fit_time_window=None,
        return_data=True,

        fit_model=runner.FIT_MODEL,
        fit_parallax=runner.FIT_PARALLAX,
        fit_defaults=None,
        fit_bounds=getattr(
            runner,
            "FIT_BOUNDS_NOPIE",
            getattr(runner, "FIT_BOUNDS", None),
        ),

        # Rubin / MAF.
        rubin_pointing_mode=runner.RUBIN_POINTING_MODE,
        rubin_cache_cell_deg=runner.RUBIN_CACHE_CELL_DEG,

        # No rechazar el evento completo por criterios globales de detección.
        # El filtro punto-a-punto por m5 se controla con apply_photometric_filter.
        apply_detection_criteria=False,
    )

    if (
        "apply_photometric_filter"
        in inspect.signature(runner.sim_fit).parameters
    ):
        kwargs["apply_photometric_filter"] = APPLY_PHOTOMETRIC_FILTER_FOR_FIT
    else:
        raise RuntimeError(
            "runner.sim_fit no acepta apply_photometric_filter. "
            "No puedo filtrar por m5 desde este workflow sin modificar sim_fit."
        )

    print("=" * 80)
    print("PHOTOMETRIC FILTER")
    print("=" * 80)
    print("apply_photometric_filter =", kwargs["apply_photometric_filter"])
    print("apply_detection_criteria =", kwargs["apply_detection_criteria"])
    print("=" * 80)

    with deterministic_rng(SEED):
        simfit_result = runner.sim_fit(**kwargs)

finally:
    runner.clear_runtime_event_context()


# ============================================================
# Verificar resultado
# ============================================================

if simfit_result["status"] != "fitted":
    raise RuntimeError(
        "El evento no terminó ajustado. "
        f"status={simfit_result['status']!r}"
    )

print("\nGuardado en:")
print(path_to_save_model)
print(path_to_save_fit)
print(path_to_save_results)


# ============================================================
# Modelo TRUE: usar EXACTAMENTE el que generó los datos
# ============================================================

true_model_obj = simfit_result["pyLIMAmodel_true"]
true_pyLIMA_parameters = simfit_result["pyLIMA_parameters_true"]
true_event_params = simfit_result["event_params"]

true_model_parameters = vector_from_pylima_parameters(
    true_model_obj,
    true_pyLIMA_parameters,
)


# ============================================================
# Modelo FIT
# ============================================================

fit_rr = simfit_result["fit_rr"]
fit_model_obj = simfit_result["pyLIMAmodel_rr"]
fit_model_parameters = np.asarray(
    fit_rr.fit_results["best_model"],
    dtype=float,
)


# ============================================================
# Diagnóstico: puntos después del filtro fotométrico
# ============================================================

n_true_points = count_points_by_telescope(
    true_model_obj,
    "TRUE model points after photometric filtering",
)

n_fit_points = count_points_by_telescope(
    fit_model_obj,
    "FIT model points after photometric filtering",
)


# ============================================================
# Validación geométrica
# ============================================================

true_ra = float(true_model_obj.event.ra)
true_dec = float(true_model_obj.event.dec)
fit_ra = float(fit_model_obj.event.ra)
fit_dec = float(fit_model_obj.event.dec)

print("\n" + "=" * 70)
print("GEOMETRY CHECK")
print("=" * 70)
print(f"catalog/source : RA={float(base_row['ra']):.10f}, Dec={float(base_row['dec']):.10f}")
print(f"simulation     : RA={true_ra:.10f}, Dec={true_dec:.10f}")
print(f"fit            : RA={fit_ra:.10f}, Dec={fit_dec:.10f}")
print("true parallax  :", true_model_obj.parallax_model)
print("fit parallax   :", fit_model_obj.parallax_model)

assert np.isclose(true_ra, fit_ra, rtol=0.0, atol=1e-10), (
    f"RA simulation/fit distinta: {true_ra} != {fit_ra}"
)

assert np.isclose(true_dec, fit_dec, rtol=0.0, atol=1e-10), (
    f"Dec simulation/fit distinta: {true_dec} != {fit_dec}"
)

if (
    bool(runner.USE_RUBIN)
    and not bool(runner.USE_ROMAN)
    and str(runner.RUBIN_POINTING_MODE).lower() == "source"
):
    assert np.isclose(
        true_ra,
        float(base_row["ra"]),
        rtol=0.0,
        atol=1e-10,
    )

    assert np.isclose(
        true_dec,
        float(base_row["dec"]),
        rtol=0.0,
        atol=1e-10,
    )




%matplotlib widget

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from contextlib import redirect_stdout
from matplotlib.patches import Circle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

from pyLIMA import event, telescopes
from pyLIMA.models import FSPL_model
from set_model_pyLIMA import build_pyLIMA_model

ZP_PYLIMA = 27.4


# ============================================================
# Helpers generales
# ============================================================

def _has(params, name):
    if isinstance(params, dict):
        return name in params

    if hasattr(params, name):
        return True

    try:
        params[name]
        return True
    except Exception:
        return False


def _val(params, name, default=None):
    if isinstance(params, dict):
        return params.get(name, default)

    if hasattr(params, name):
        return getattr(params, name)

    try:
        return params[name]
    except Exception:
        return default


def _to_numpy(x):
    return np.asarray(
        x.value if hasattr(x, "value") else x,
        dtype=float,
    )


def _arr(x):
    return np.asarray(
        getattr(x, "value", x),
        dtype=float,
    )

ZP_PYLIMA = 27.4

ZP_RUBIN = {
    "u": 27.03,
    "g": 28.38,
    "r": 28.16,
    "i": 27.85,
    "z": 27.46,
    "y": 26.68,
    "W149": 27.615,
}


def _band_zp(band):
    band = str(band)

    if band in ZP_RUBIN:
        return float(ZP_RUBIN[band])

    # fallback por si aparece Simulation u otro nombre interno
    return float(ZP_PYLIMA)


def _mag_to_flux_band(mag, band):
    zp = _band_zp(band)

    return 10.0 ** (
        (zp - np.asarray(mag, dtype=float)) / 2.5
    )


def _flux_to_mag_band(flux, band):
    zp = _band_zp(band)

    flux = np.asarray(flux, dtype=float)

    mag = np.full(
        flux.shape,
        np.nan,
        dtype=float,
    )

    ok = np.isfinite(flux) & (flux > 0.0)

    mag[ok] = zp - 2.5 * np.log10(flux[ok])

    return mag


def _magerr_to_fluxerr(mag, err_mag, band):
    """
    Propaga error de magnitud a error de flujo usando el zero-point de esa banda.
    """

    flux = _mag_to_flux_band(
        mag,
        band,
    )

    return (
        np.log(10.0)
        / 2.5
        * flux
        * np.asarray(err_mag, dtype=float)
    )


def _fluxerr_to_magerr(flux, err_flux):
    """
    Propaga error de flujo a error de magnitud.
    """

    flux = np.asarray(flux, dtype=float)
    err_flux = np.asarray(err_flux, dtype=float)

    err_mag = np.full(
        flux.shape,
        np.nan,
        dtype=float,
    )

    ok = (
        np.isfinite(flux)
        & np.isfinite(err_flux)
        & (flux > 0.0)
        & (err_flux >= 0.0)
    )

    err_mag[ok] = (
        2.5
        / np.log(10.0)
        * err_flux[ok]
        / flux[ok]
    )

    return err_mag


def _get_flux_pair(pyparams, band):
    fs = float(_val(pyparams, f"fsource_{band}"))

    if _has(pyparams, f"fblend_{band}"):
        fb = float(_val(pyparams, f"fblend_{band}"))

    elif _has(pyparams, f"ftotal_{band}"):
        fb = float(
            _val(pyparams, f"ftotal_{band}")
            - _val(pyparams, f"fsource_{band}")
        )

    else:
        raise KeyError(
            f"No encontré fblend_{band} ni ftotal_{band}."
        )

    return fs, fb


def _vector_from_pyparams(model_obj, pyparams):
    """
    Reconstruye el vector de parámetros en el orden real esperado por pyLIMA.
    Se usa para la curva densa, no para la trayectoria estilo widget.
    """

    values = []
    missing = []

    for name, _ in sorted(
        model_obj.model_dictionnary.items(),
        key=lambda x: x[1],
    ):

        if _has(pyparams, name):
            values.append(float(_val(pyparams, name)))
            continue

        if name.startswith("fblend_"):
            band = name.split("_")[-1]

            if (
                _has(pyparams, f"ftotal_{band}")
                and _has(pyparams, f"fsource_{band}")
            ):
                values.append(
                    float(
                        _val(pyparams, f"ftotal_{band}")
                        - _val(pyparams, f"fsource_{band}")
                    )
                )
                continue

        if name.startswith("ftotal_"):
            band = name.split("_")[-1]

            if (
                _has(pyparams, f"fblend_{band}")
                and _has(pyparams, f"fsource_{band}")
            ):
                values.append(
                    float(
                        _val(pyparams, f"fsource_{band}")
                        + _val(pyparams, f"fblend_{band}")
                    )
                )
                continue

        missing.append(name)

    if missing:
        raise KeyError(
            f"No pude reconstruir el vector. Faltan: {missing}"
        )

    return np.asarray(values, dtype=float)


def _active_bands(model_obj):
    bands = []

    for tel in model_obj.event.telescopes:
        if tel.lightcurve is None or len(tel.lightcurve) == 0:
            continue

        bands.append(tel.name)

    return bands


def _derive_fit_fluxes(fit_model_obj, fit_pyparams):
    """
    Completa fblend si pyLIMA lo deriva internamente.

    Importante:
    estos flujos NO se usan para alinear los datos.
    """

    for tel in fit_model_obj.event.telescopes:
        if tel.lightcurve is None or len(tel.lightcurve) == 0:
            continue

        A = fit_model_obj.model_magnification(
            tel,
            fit_pyparams,
        )

        fit_model_obj.derive_telescope_flux(
            tel,
            fit_pyparams,
            A,
        )

    return fit_pyparams


# ============================================================
# Diagnóstico de flujos del fit
# ============================================================

def check_fit_fluxes_are_physical(
    fit_model_obj,
    fit_model_parameters,
):
    fit_pyparams = fit_model_obj.compute_pyLIMA_parameters(
        fit_model_parameters,
    )

    fit_pyparams = _derive_fit_fluxes(
        fit_model_obj,
        fit_pyparams,
    )

    bad = []

    print("=" * 80)
    print("FIT flux diagnostics")
    print("=" * 80)

    for tel in fit_model_obj.event.telescopes:

        if tel.lightcurve is None or len(tel.lightcurve) == 0:
            continue

        band = tel.name

        fs, fb = _get_flux_pair(
            fit_pyparams,
            band,
        )

        ft = fs + fb
        frac = fs / ft if ft != 0.0 else np.nan

        print(
            f"{band:3s}  "
            f"fsource={fs: .6e}  "
            f"ftotal={ft: .6e}  "
            f"fblend={fb: .6e}  "
            f"fsource/ftotal={frac: .6e}"
        )

        if (
            not np.isfinite(fs)
            or not np.isfinite(fb)
            or not np.isfinite(ft)
            or fs <= 0.0
            or ft <= 0.0
            or fb < 0.0
            or fs > ft
        ):
            bad.append(
                {
                    "band": band,
                    "fsource": fs,
                    "fblend": fb,
                    "ftotal": ft,
                    "fsource_over_ftotal": frac,
                }
            )

    if bad:
        print("=" * 80)
        print("WARNING: el fit tiene flujos no físicos.")
        print("El plot usará flujos TRUE para alinear las bandas.")
        print("=" * 80)

    return pd.DataFrame(bad)


# ============================================================
# Paralaje para curva densa
# ============================================================

def _is_no_parallax(parallax_model):
    if parallax_model is None:
        return True

    return str(parallax_model[0]).lower() == "none"


def _force_parallax_reference_to_t0(
    parallax_model,
    t0_reference,
):
    """
    Fuerza que el epoch de referencia del paralaje sea t0_reference.

    Si no hay paralaje, devuelve ['None', 0.0].
    """

    if _is_no_parallax(parallax_model):
        return ["None", 0.0]

    return [
        parallax_model[0],
        float(t0_reference),
    ]


def _parallax_to_label(parallax_model):
    if _is_no_parallax(parallax_model):
        return "No parallax"

    return f"{parallax_model[0]} parallax"


# ============================================================
# Modelo denso para curva de luz
# ============================================================

def _make_dense_model_for_lightcurve(
    t_dense,
    band,
    ra,
    dec,
    model_name,
    parallax_model,
    pyparams,
):
    """
    Construye un modelo pyLIMA denso para calcular A(t).

    Esta función se usa para la curva de luz, no para el inset.
    """

    lc_dense = np.column_stack(
        [
            t_dense,
            np.ones_like(t_dense) * 20.0,
            np.ones_like(t_dense) * 0.01,
        ]
    )

    e_dense = event.Event(
        ra=float(ra),
        dec=float(dec),
    )

    e_dense.name = f"dense_{model_name}_{band}"

    tel_dense = telescopes.Telescope(
        name=band,
        camera_filter=band,
        lightcurve=lc_dense.astype(float),
        lightcurve_names=["time", "mag", "err_mag"],
        lightcurve_units=["JD", "mag", "mag"],
        location="Earth",
    )

    tel_dense.ld_gamma = 0.0

    e_dense.telescopes.append(
        tel_dense,
    )

    e_dense.check_event()

    use_parallax = not _is_no_parallax(
        parallax_model,
    )

    t0_parallax = float(
        parallax_model[1],
    )

    dense_model = build_pyLIMA_model(
        pyLIMA_event=e_dense,
        model=model_name,
        use_parallax=use_parallax,
        t0_parallax=t0_parallax,
        origin=None,
        random_origin=False,
        blend_flux_parameter="ftotal",
    )

    dense_parameters = _vector_from_pyparams(
        dense_model,
        pyparams,
    )

    dense_pyparams = dense_model.compute_pyLIMA_parameters(
        dense_parameters,
    )

    A_dense = dense_model.model_magnification(
        dense_model.event.telescopes[0],
        dense_pyparams,
    )

    return dense_model, dense_parameters, dense_pyparams, A_dense


# ============================================================
# Trayectoria estilo widget: SOLO FSPL_model.FSPLmodel
# ============================================================

def _build_sim_event_widget_style(
    t,
    ra,
    dec,
    mag0=19.0,
    emag=1e-6,
    filt="g",
):
    """
    Igual que en tu widget:
    Event ficticio + Telescope ficticio.

    La curva ficticia no se usa como dato.
    Solo da la grilla temporal para sources_trajectory().
    """

    ev = event.Event()
    ev.name = "FSPL widget-style trajectory"
    ev.ra = float(ra)
    ev.dec = float(dec)

    lc = np.c_[
        t,
        np.full_like(t, mag0),
        np.full_like(t, emag),
    ]

    tel = telescopes.Telescope(
        name="Simulation",
        camera_filter=filt,
        lightcurve=lc.astype(float),
        lightcurve_names=[
            "time",
            "mag",
            "err_mag",
        ],
        lightcurve_units=[
            "JD",
            "mag",
            "mag",
        ],
        location="Earth",
    )

    tel.ld_gamma = 0.0

    ev.telescopes.append(
        tel,
    )

    return ev


def _make_pylima_parameters_widget_style(
    model,
    values,
):
    """
    Igual que en tu widget:
    construye el vector siguiendo model.model_dictionnary.keys().
    """

    vector = []

    for parameter_name in model.model_dictionnary.keys():
        vector.append(
            values.get(parameter_name, None),
        )

    return model.compute_pyLIMA_parameters(
        vector,
    )


def _compute_fspl_trajectory_widget_style(
    t,
    ra,
    dec,
    values,
    parallax,
    filt="g",
):
    """
    Calcula trayectoria usando exactamente el método del widget:

        FSPL_model.FSPLmodel(ev, parallax=...)
        model.sources_trajectory(tel, params, data_type="photometry")

    No usa build_pyLIMA_model.
    No usa fallback.
    """

    ev = _build_sim_event_widget_style(
        t=t,
        ra=ra,
        dec=dec,
        filt=filt,
    )

    tel = ev.telescopes[0]

    if str(parallax[0]).lower() == "none":
        model = FSPL_model.FSPLmodel(
            ev,
            parallax=parallax,
        )
    else:
        with redirect_stdout(io.StringIO()):
            model = FSPL_model.FSPLmodel(
                ev,
                parallax=parallax,
            )

    params = _make_pylima_parameters_widget_style(
        model,
        values,
    )

    traj = model.sources_trajectory(
        tel,
        params,
        data_type="photometry",
    )

    x = _arr(traj[0])
    y = _arr(traj[1])

    return {
        "x": x,
        "y": y,
        "t": np.asarray(t, dtype=float),
        "model": model,
        "params": params,
        "event": ev,
        "telescope": tel,
        "parallax": parallax,
        "values": values,
    }


def _build_trajectory_specs_widget_style(
    true_pyparams,
    fit_pyparams,
    true_model_obj,
    fit_model_obj,
    t_traj,
    ra,
    dec,
    reference_band="g",
    show_true_no_parallax_reference=True,
):
    """
    Devuelve trajectory_specs y traj_data usando el código estilo widget.
    """

    t0_true = float(_val(true_pyparams, "t0"))
    u0_true = float(_val(true_pyparams, "u0"))
    tE_true = float(_val(true_pyparams, "tE"))
    rho_true = float(_val(true_pyparams, "rho"))

    piEN_true = (
        float(_val(true_pyparams, "piEN"))
        if _has(true_pyparams, "piEN")
        else 0.0
    )

    piEE_true = (
        float(_val(true_pyparams, "piEE"))
        if _has(true_pyparams, "piEE")
        else 0.0
    )

    true_values_parallax = {
        "t0": t0_true,
        "u0": u0_true,
        "tE": tE_true,
        "rho": rho_true,
        "piEN": piEN_true,
        "piEE": piEE_true,
    }

    true_parallax = _compute_fspl_trajectory_widget_style(
        t=t_traj,
        ra=ra,
        dec=dec,
        values=true_values_parallax,
        parallax=["Full", t0_true],
        filt=reference_band,
    )

    trajectory_specs = [
        {
            "x": true_parallax["x"],
            "y": true_parallax["y"],
            "label": "True",
            "color": "k",
            "ls": "-",
            "lw": 1.9,
            "draw_rho": True,
        }
    ]

    true_no_parallax = None

    if show_true_no_parallax_reference:

        true_values_no_parallax = {
            "t0": t0_true,
            "u0": u0_true,
            "tE": tE_true,
            "rho": rho_true,
        }

        true_no_parallax = _compute_fspl_trajectory_widget_style(
            t=t_traj,
            ra=ra,
            dec=dec,
            values=true_values_no_parallax,
            parallax=["None", t0_true],
            filt=reference_band,
        )

        trajectory_specs.append(
            {
                "x": true_no_parallax["x"],
                "y": true_no_parallax["y"],
                "label": "True params, no parallax",
                "color": "C0",
                "ls": ":",
                "lw": 1.8,
                "draw_rho": False,
            }
        )

    fit_traj = None
    fit_values = None

    if fit_pyparams is not None:

        fit_t0 = float(_val(fit_pyparams, "t0"))
        fit_u0 = float(_val(fit_pyparams, "u0"))
        fit_tE = float(_val(fit_pyparams, "tE"))

        fit_rho = (
            float(_val(fit_pyparams, "rho"))
            if _has(fit_pyparams, "rho")
            else rho_true
        )

        fit_parallax_model = getattr(
            fit_model_obj,
            "parallax_model",
            ["None", fit_t0],
        )

        if str(fit_parallax_model[0]).lower() == "none":

            fit_values = {
                "t0": fit_t0,
                "u0": fit_u0,
                "tE": fit_tE,
                "rho": fit_rho,
            }

            fit_parallax_for_plot = [
                "None",
                fit_t0,
            ]

        else:

            fit_values = {
                "t0": fit_t0,
                "u0": fit_u0,
                "tE": fit_tE,
                "rho": fit_rho,
                "piEN": float(_val(fit_pyparams, "piEN")),
                "piEE": float(_val(fit_pyparams, "piEE")),
            }

            fit_parallax_for_plot = [
                fit_parallax_model[0],
                fit_t0,
            ]

        fit_traj = _compute_fspl_trajectory_widget_style(
            t=t_traj,
            ra=ra,
            dec=dec,
            values=fit_values,
            parallax=fit_parallax_for_plot,
            filt=reference_band,
        )

        trajectory_specs.append(
            {
                "x": fit_traj["x"],
                "y": fit_traj["y"],
                "label": "Fit",
                "color": "purple",
                "ls": "--",
                "lw": 1.9,
                "draw_rho": False,
            }
        )

    traj_data = {
        "t": t_traj,
        "true_parallax": true_parallax,
        "true_no_parallax": true_no_parallax,
        "fit": fit_traj,
        "true_values": true_values_parallax,
        "fit_values": fit_values,
    }

    print("=" * 80)
    print("WIDGET-STYLE TRAJECTORY CHECK")
    print("=" * 80)
    print("RA, Dec =", ra, dec)
    print("TRUE parallax used =", ["Full", t0_true])
    print("TRUE t0, u0, tE, rho =", t0_true, u0_true, tE_true, rho_true)
    print("TRUE piEN, piEE =", piEN_true, piEE_true)
    print(
        "TRUE parallax min u =",
        np.nanmin(
            np.hypot(
                true_parallax["x"],
                true_parallax["y"],
            )
        ),
    )

    if true_no_parallax is not None:
        print(
            "TRUE no-parallax min u =",
            np.nanmin(
                np.hypot(
                    true_no_parallax["x"],
                    true_no_parallax["y"],
                )
            ),
        )

    if fit_traj is not None:
        print("FIT parallax used =", fit_traj["parallax"])
        print("FIT values =", fit_values)
        print(
            "FIT min u on TRUE-time grid =",
            np.nanmin(
                np.hypot(
                    fit_traj["x"],
                    fit_traj["y"],
                )
            ),
        )

    print("=" * 80)

    return trajectory_specs, traj_data


# ============================================================
# Inset trajectory
# ============================================================

def _add_direction_arrows(
    ax,
    x,
    y,
    color,
    n_arrows=3,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 4:
        return

    idxs = np.linspace(
        1,
        len(x) - 2,
        n_arrows,
        dtype=int,
    )

    for idx in idxs:

        dx = x[idx + 1] - x[idx - 1]
        dy = y[idx + 1] - y[idx - 1]

        norm = np.hypot(
            dx,
            dy,
        )

        if not np.isfinite(norm) or norm == 0:
            continue

        scale = 0.08

        ax.annotate(
            "",
            xy=(
                x[idx] + scale * dx / norm,
                y[idx] + scale * dy / norm,
            ),
            xytext=(
                x[idx],
                y[idx],
            ),
            arrowprops=dict(
                arrowstyle="->",
                mutation_scale=11,
                color=color,
                lw=1.1,
            ),
        )

def _add_trajectory_inset(
    ax,
    trajectory_specs,
    rho_true=None,
    loc="upper left",
    width="35%",
    height="45%",
):
    """
    Agrega inset usando trajectory_specs ya calculado estilo widget.

    En la leyenda del inset solo deja:
        - theta_E
        - Lens

    Las trayectorias True/Fit se grafican pero no entran en la leyenda.
    """

    axins = inset_axes(
        ax,
        width=width,
        height=height,
        loc=loc,
        borderpad=1.2,
    )

    # ------------------------------------------------------------
    # Einstein ring
    # ------------------------------------------------------------

    phi = np.linspace(
        0.0,
        2.0 * np.pi,
        400,
    )

    axins.plot(
        np.cos(phi),
        np.sin(phi),
        color="0.5",
        ls=":",
        lw=1.0,
        alpha=0.8,
        label=r"$\theta_E$",
    )

    # ------------------------------------------------------------
    # Lens
    # ------------------------------------------------------------

    axins.scatter(
        [0.0],
        [0.0],
        marker="+",
        s=90,
        color="k",
        zorder=10,
        label="Lens",
    )

    all_x = [
        np.array([-1.0, 1.0])
    ]

    all_y = [
        np.array([-1.0, 1.0])
    ]

    # ------------------------------------------------------------
    # Trayectorias True/Fit
    # ------------------------------------------------------------

    for spec in trajectory_specs:

        x = np.asarray(spec["x"], dtype=float)
        y = np.asarray(spec["y"], dtype=float)

        color = spec.get("color", None)
        ls = spec.get("ls", "-")
        lw = spec.get("lw", 1.8)

        # Importante:
        # label="_nolegend_" evita que True/Fit entren en la leyenda del inset.
        axins.plot(
            x,
            y,
            color=color,
            ls=ls,
            lw=lw,
            label="_nolegend_",
        )

        _add_direction_arrows(
            axins,
            x,
            y,
            color=color,
            n_arrows=3,
        )

        u = np.hypot(
            x,
            y,
        )

        if np.any(np.isfinite(u)):

            i_min = int(
                np.nanargmin(u)
            )

            axins.scatter(
                [x[i_min]],
                [y[i_min]],
                s=24,
                color=color,
                zorder=8,
                label="_nolegend_",
            )

            if spec.get("draw_rho", False):
                if rho_true is not None and np.isfinite(rho_true) and rho_true > 0:

                    source_circle = Circle(
                        (x[i_min], y[i_min]),
                        rho_true,
                        fill=False,
                        color=color,
                        lw=1.0,
                        alpha=0.9,
                    )

                    axins.add_patch(
                        source_circle,
                    )

        all_x.append(
            x[np.isfinite(x)]
        )

        all_y.append(
            y[np.isfinite(y)]
        )

    # ------------------------------------------------------------
    # Límites
    # ------------------------------------------------------------

    all_x = np.concatenate(
        all_x,
    )

    all_y = np.concatenate(
        all_y,
    )

    x_min, x_max = np.nanmin(all_x), np.nanmax(all_x)
    y_min, y_max = np.nanmin(all_y), np.nanmax(all_y)

    cx = 0.5 * (x_min + x_max)
    cy = 0.5 * (y_min + y_max)

    half = 0.58 * max(
        x_max - x_min,
        y_max - y_min,
        2.0,
    )

    axins.set_xlim(
        cx - half,
        cx + half,
    )

    axins.set_ylim(
        cy - half,
        cy + half,
    )

    axins.axhline(
        0,
        color="k",
        lw=0.5,
        alpha=0.25,
    )

    axins.axvline(
        0,
        color="k",
        lw=0.5,
        alpha=0.25,
    )

    axins.set_aspect(
        "equal",
        adjustable="box",
    )

    axins.set_xlabel(
        r"$u_x$",
        fontsize=8,
    )

    axins.set_ylabel(
        r"$u_y$",
        fontsize=8,
    )

    axins.tick_params(
        labelsize=7,
    )

    axins.grid(
        alpha=0.20,
    )

    # Solo quedan en la leyenda theta_E y Lens.
    axins.legend(
        fontsize=7,
        loc="best",
        framealpha=0.85,
    )


    return axins


# ============================================================
# Plot principal
# ============================================================

def plot_event_aligned_true_fit_with_widget_style_trajectory(
    true_model_obj,
    true_model_parameters,
    fit_model_obj,
    fit_model_parameters,
    true_params=None,
    reference_band="g",
    n_dense=10000,
    n_tE=2,
    inset_n_tE=3.5,
    true_model_name=None,
    fit_model_name=None,
    skip_bad_aligned_points=True,
    inset_loc="lower left",
    force_true_parallax_reference_to_t0=True,
    show_true_no_parallax_reference=True,
):
    """
    Curva de luz + residuales + inset de trayectoria.

    Curva de luz:
        - se grafica en escala de flujo TRUE de reference_band.

    Trayectoria:
        - se calcula exclusivamente con el método estilo widget:
          FSPL_model.FSPLmodel(...).sources_trajectory(...).
    """

    if true_model_name is None:
        true_model_name = runner.MODEL

    if fit_model_name is None:
        fit_model_name = runner.FIT_MODEL

    true_pyparams = true_model_obj.compute_pyLIMA_parameters(
        true_model_parameters,
    )

    fit_pyparams = fit_model_obj.compute_pyLIMA_parameters(
        fit_model_parameters,
    )

    fit_pyparams = _derive_fit_fluxes(
        fit_model_obj,
        fit_pyparams,
    )

    t0 = float(
        _val(true_pyparams, "t0")
    )

    tE = float(
        _val(true_pyparams, "tE")
    )

    fit_t0 = float(
        _val(fit_pyparams, "t0")
    )

    fit_tE = float(
        _val(fit_pyparams, "tE")
    )

    print("t0 usado para graficar =", t0)
    print("tE usado para graficar =", tE)
    print("t0 fit                 =", fit_t0)
    print("tE fit                 =", fit_tE)

    if true_params is not None:
        print("t0 en true_params      =", float(_val(true_params, "t0")))
        print("tE en true_params      =", float(_val(true_params, "tE")))

    ra = float(
        true_model_obj.event.ra,
    )

    dec = float(
        true_model_obj.event.dec,
    )

    active_bands = _active_bands(
        fit_model_obj,
    )

    if reference_band not in active_bands:
        raise ValueError(
            f"{reference_band=} no está activa. "
            f"Bandas activas: {active_bands}"
        )

    # ------------------------------------------------------------
    # Parallax model para la curva densa
    # ------------------------------------------------------------

    if force_true_parallax_reference_to_t0:
        true_parallax_model_for_plot = _force_parallax_reference_to_t0(
            true_model_obj.parallax_model,
            t0,
        )
    else:
        true_parallax_model_for_plot = true_model_obj.parallax_model

    if _is_no_parallax(fit_model_obj.parallax_model):
        fit_parallax_model_for_plot = ["None", 0.0]
    else:
        fit_parallax_model_for_plot = _force_parallax_reference_to_t0(
            fit_model_obj.parallax_model,
            fit_t0,
        )

    print("=" * 80)
    print("PARALLAX REFERENCE CHECK")
    print("=" * 80)
    print("true original parallax_model =", true_model_obj.parallax_model)
    print("true plot parallax_model     =", true_parallax_model_for_plot)
    print("fit original parallax_model  =", fit_model_obj.parallax_model)
    print("fit plot parallax_model      =", fit_parallax_model_for_plot)
    print("=" * 80)

    # ------------------------------------------------------------
    # Flujos TRUE de referencia
    # ------------------------------------------------------------

    fsource_ref, fblend_ref = _get_flux_pair(
        true_pyparams,
        reference_band,
    )

    if fsource_ref <= 0.0 or not np.isfinite(fsource_ref):
        raise RuntimeError(
            f"fsource TRUE inválido en {reference_band}: {fsource_ref}"
        )

    # ------------------------------------------------------------
    # Curva de luz densa
    # ------------------------------------------------------------

    t_dense = np.linspace(
        t0 - n_tE * tE,
        t0 + n_tE * tE,
        int(n_dense),
    )

    true_dense_model, _, true_dense_pyparams, A_true_dense = _make_dense_model_for_lightcurve(
        t_dense=t_dense,
        band=reference_band,
        ra=ra,
        dec=dec,
        model_name=true_model_name,
        parallax_model=true_parallax_model_for_plot,
        pyparams=true_pyparams,
    )

    fit_dense_model, _, fit_dense_pyparams, A_fit_dense = _make_dense_model_for_lightcurve(
        t_dense=t_dense,
        band=reference_band,
        ra=ra,
        dec=dec,
        model_name=fit_model_name,
        parallax_model=fit_parallax_model_for_plot,
        pyparams=fit_pyparams,
    )

    print("=" * 80)
    print("DENSE MODEL CHECK")
    print("=" * 80)
    print("true dense class          =", type(true_dense_model))
    print("true dense parallax_model =", true_dense_model.parallax_model)
    print("fit dense class           =", type(fit_dense_model))
    print("fit dense parallax_model  =", fit_dense_model.parallax_model)
    print("=" * 80)

    m_true_dense = _flux_to_mag_band(
        fsource_ref * A_true_dense + fblend_ref,
        reference_band,
    )
    
    m_fit_dense = _flux_to_mag_band(
        fsource_ref * A_fit_dense + fblend_ref,
        reference_band,
    )

    # ------------------------------------------------------------
    # Datos alineados con flujos TRUE
    # ------------------------------------------------------------

    rows = []
    skipped = []

    for tel in fit_model_obj.event.telescopes:

        if tel.lightcurve is None or len(tel.lightcurve) == 0:
            continue

        band = tel.name

        A_fit_tel = fit_model_obj.model_magnification(
            tel,
            fit_pyparams,
        )

        fsource_tel, fblend_tel = _get_flux_pair(
            true_pyparams,
            band,
        )

        if fsource_tel <= 0.0 or not np.isfinite(fsource_tel):
            skipped.append(
                {
                    "band": band,
                    "reason": "invalid_true_fsource",
                    "fsource_true": fsource_tel,
                }
            )
            continue

        time = _to_numpy(
            tel.lightcurve["time"],
        )

        mag = _to_numpy(
            tel.lightcurve["mag"],
        )

        err_mag = _to_numpy(
            tel.lightcurve["err_mag"],
        )
        # ------------------------------------------------------------
        # Observed flux in the native zero-point of this band
        # ------------------------------------------------------------
        
        flux_obs = _mag_to_flux_band(
            mag,
            band,
        )
        
        err_flux_obs = _magerr_to_fluxerr(
            mag,
            err_mag,
            band,
        )
        
        # ------------------------------------------------------------
        # Convert observed flux to an equivalent magnification
        # using TRUE fluxes of the same band
        # ------------------------------------------------------------
        
        A_obs_equiv = (
            flux_obs - fblend_tel
        ) / fsource_tel
        
        err_A_obs_equiv = (
            err_flux_obs / fsource_tel
        )
        
        # ------------------------------------------------------------
        # Move that equivalent magnification to the reference band
        # ------------------------------------------------------------
        
        flux_aligned = (
            fsource_ref * A_obs_equiv
            + fblend_ref
        )
        
        err_flux_aligned = (
            fsource_ref * err_A_obs_equiv
        )
        
        mag_aligned = _flux_to_mag_band(
            flux_aligned,
            reference_band,
        )
        
        err_mag_aligned = _fluxerr_to_magerr(
            flux_aligned,
            err_flux_aligned,
        )
        
        # ------------------------------------------------------------
        # Fit model evaluated in the reference-band TRUE flux scale
        # ------------------------------------------------------------
        
        flux_fit_aligned = (
            fsource_ref * A_fit_tel
            + fblend_ref
        )
        
        mag_fit_aligned = _flux_to_mag_band(
            flux_fit_aligned,
            reference_band,
        )
        
        residual = (
            mag_aligned
            - mag_fit_aligned
        )

        for k in range(len(time)):

            row = {
        "band": band,
        "time": float(time[k]),
        "t_minus_t0": float(time[k] - t0),
        "mag": float(mag[k]),
        "mag_aligned": float(mag_aligned[k]),
        "err_mag": float(err_mag_aligned[k]),
        "err_mag_original": float(err_mag[k]),
        "fit_mag_aligned": float(mag_fit_aligned[k]),
        "residual": float(residual[k]),
        "A_obs_equiv": float(A_obs_equiv[k]),
        "A_fit": float(A_fit_tel[k]),
        "fsource_true_band": float(fsource_tel),
        "fblend_true_band": float(fblend_tel),
    }
            bad_point = (
                not np.isfinite(row["mag_aligned"])
                or not np.isfinite(row["residual"])
                or row["mag_aligned"] < 0.0
                or row["mag_aligned"] > 40.0
            )

            if bad_point and skip_bad_aligned_points:
                skipped.append(
                    {
                        **row,
                        "reason": "bad_aligned_point",
                    }
                )
                continue

            rows.append(
                row,
            )

    aligned_data = pd.DataFrame(
        rows,
    )

    skipped_data = pd.DataFrame(
        skipped,
    )

    if len(skipped_data) > 0:
        print("=" * 80)
        print(f"Skipped aligned points: {len(skipped_data)}")
        print("=" * 80)

        try:
            display(skipped_data.head(20))
        except NameError:
            print(skipped_data.head(20))

    if len(aligned_data) == 0:
        raise RuntimeError(
            "No quedó ningún punto válido para graficar."
        )

    # ------------------------------------------------------------
    # Trayectorias: widget-style, NO build_pyLIMA_model
    # ------------------------------------------------------------

    t_traj = np.linspace(
        t0 - inset_n_tE * tE,
        t0 + inset_n_tE * tE,
        int(n_dense),
    )

    trajectory_specs, traj_data = _build_trajectory_specs_widget_style(
        true_pyparams=true_pyparams,
        fit_pyparams=fit_pyparams,
        true_model_obj=true_model_obj,
        fit_model_obj=fit_model_obj,
        t_traj=t_traj,
        ra=ra,
        dec=dec,
        reference_band=reference_band,
        show_true_no_parallax_reference=show_true_no_parallax_reference,
    )

    rho_true = (
        float(_val(true_pyparams, "rho"))
        if _has(true_pyparams, "rho")
        else None
    )

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(10, 7),
        dpi=130,
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
    )

    ax, ax_res = axes

    ax.plot(
        t_dense,
        m_true_dense,
        "k-",
        lw=2.0,
        label=f"{true_model_name} + {_parallax_to_label(true_parallax_model_for_plot)}",
    )

    ax.plot(
        t_dense,
        m_fit_dense,
        "--",
        color="purple",
        lw=2.0,
        label=f"{fit_model_name} fit + {_parallax_to_label(fit_parallax_model_for_plot)}",
    )

    marker_map = {
        "u": "o",
        "g": "s",
        "r": "d",
        "i": "^",
        "z": "*",
        "y": "v",
        "W149": "x",
    }

    for band, group in aligned_data.groupby("band"):

        group = group.sort_values(
            "time",
        )

        ax.errorbar(
            group["time"],
            group["mag_aligned"],
            yerr=group["err_mag"],
            fmt=marker_map.get(band, "."),
            ms=4,
            linestyle="none",
            alpha=0.75,
            label=band,
        )

        ax_res.errorbar(
            group["time"],
            group["residual"],
            yerr=group["err_mag"],
            fmt=marker_map.get(band, "."),
            ms=4,
            linestyle="none",
            alpha=0.75,
        )

    ax.axvline(
        t0,
        ls="--",
        color="0.5",
        label=r"$t_{0,\mathrm{true}}$",
    )

    ax.axvline(
        fit_t0,
        ls=":",
        color="purple",
        alpha=0.7,
        label=r"$t_{0,\mathrm{fit}}$",
    )

    ax.axvspan(
        t0 - tE,
        t0 + tE,
        alpha=0.15,
        label=r"$t_{0,\mathrm{true}}\pm t_E$",
    )

    ax_res.axhline(
        0.0,
        color="k",
        lw=1,
        alpha=0.5,
    )

    ax.set_xlim(
        t0 - n_tE * tE,
        t0 + n_tE * tE,
    )

    ax.invert_yaxis()
    ax_res.invert_yaxis()

    ax.set_ylabel(
        f"Aligned magnitude ({reference_band}, TRUE flux scale)"
    )

    ax_res.set_ylabel(
        "data - fit [mag]"
    )

    ax_res.set_xlabel(
        "JD"
    )

    ax.grid(
        alpha=0.25,
    )

    ax_res.grid(
        alpha=0.25,
    )

    ax_traj = _add_trajectory_inset(
        ax=ax,
        trajectory_specs=trajectory_specs,
        rho_true=rho_true,
        loc=inset_loc,
        width="35%",
        height="45%",
    )

    handles, labels = ax.get_legend_handles_labels()
    seen = {}

    for h, lab in zip(handles, labels):
        if lab and not lab.startswith("_") and lab not in seen:
            seen[lab] = h

    #ax.legend(
    #    seen.values(),
    #    seen.keys(),
    #    fontsize=8,
    #    ncol=3,
    #    framealpha=0.9,
    #    loc="best",
    #)

    ax.legend(seen.values(),seen.keys(),shadow=True, fontsize='large',
              bbox_to_anchor=(0, 1.02, 1, 0.2), loc="lower left",
              mode="expand", borderaxespad=0, ncol=3, numpoints=1)

    fig.tight_layout()

    dense = {
        "t_dense": t_dense,
        "m_true_dense": m_true_dense,
        "m_fit_dense": m_fit_dense,
        "A_true_dense": A_true_dense,
        "A_fit_dense": A_fit_dense,
        "t_traj": t_traj,
        "traj_data": traj_data,
        "true_traj_x": traj_data["true_parallax"]["x"],
        "true_traj_y": traj_data["true_parallax"]["y"],
        "fit_traj_x": (
            None if traj_data["fit"] is None
            else traj_data["fit"]["x"]
        ),
        "fit_traj_y": (
            None if traj_data["fit"] is None
            else traj_data["fit"]["y"]
        ),
        "true_no_par_traj_x": (
            None if traj_data["true_no_parallax"] is None
            else traj_data["true_no_parallax"]["x"]
        ),
        "true_no_par_traj_y": (
            None if traj_data["true_no_parallax"] is None
            else traj_data["true_no_parallax"]["y"]
        ),
        "t0": t0,
        "tE": tE,
        "fit_t0": fit_t0,
        "fit_tE": fit_tE,
        "rho_true": rho_true,
        "true_parallax_model_for_plot": true_parallax_model_for_plot,
        "fit_parallax_model_for_plot": fit_parallax_model_for_plot,
        "fsource_ref_true": fsource_ref,
        "fblend_ref_true": fblend_ref,
        "skipped_data": skipped_data,
        "ax_traj": ax_traj,
    }

    return fig, axes, aligned_data, dense


bad_fluxes = check_fit_fluxes_are_physical(
    fit_model_obj,
    fit_model_parameters,
)

fig, axes, aligned_data, dense = plot_event_aligned_true_fit_with_widget_style_trajectory(
    true_model_obj=true_model_obj,
    true_model_parameters=true_model_parameters,
    fit_model_obj=fit_model_obj,
    fit_model_parameters=fit_model_parameters,
    true_params=true_event_params,
    reference_band="g",
    n_dense=10000,
    n_tE=10,
    inset_n_tE=4.0,
    true_model_name=runner.MODEL,
    fit_model_name=runner.FIT_MODEL,
    inset_loc="upper left",
    force_true_parallax_reference_to_t0=True,
    show_true_no_parallax_reference=False,
)

print("A_true min/max =", np.nanmin(dense["A_true_dense"]), np.nanmax(dense["A_true_dense"]))
print("A_fit  min/max =", np.nanmin(dense["A_fit_dense"]), np.nanmax(dense["A_fit_dense"]))

plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/home/anibal/ulensing_degenerate_models/Parallax_LSST/lsstmonts_catalog_sedighe/run_lsstmonts_catalog_sedighe_xi.py'

In [ ]:
from pathlib import Path
from pipeline_hidden_parallax import (
    SedighePipelineConfig,
    run_one_by_global_i,
)

cfg = SedighePipelineConfig(
    runner_path=Path(
        "/home/anibal/ulensing_degenerate_models/Parallax_LSST/"
        "lsstmonts_catalog_sedighe/run_lsstmonts_catalog_sedighe_xi.py"
    ),
    config_path=Path(
        "/home/anibal/ulensing_degenerate_models/Parallax_LSST/"
        "lsstmonts_catalog_sedighe/config_lsstmonts_baseline_v5p3p5.json"
    ),
    roman_rubin_dir=Path(
        "/home/anibal/microlensing/simulation_Rubin/roman_rubin"
    ),
    out_dir=Path(
        "/mnt/almacenamiento/hidden_parallax_visual_check"
    ),

    max_base_events=10,

    apply_photometric_filter=True,
    apply_detection_criteria=False,

    truth_parallax=True,
    time_window=None,
    fit_time_window=None,

    make_plots=True,
    plot_style="aligned_inset",

    reference_band="g",
    allow_reference_band_fallback=True,

    plot_n_dense=10000,
    plot_n_tE=10.0,
    plot_inset_n_tE=4.0,
    plot_inset_loc="upper left",
    plot_show_true_no_parallax_reference=False,

    # Importante: trayectoria del fit alrededor de t0_fit.
    plot_fit_trajectory_time_mode="own_fit_tE_window",

    reset_output=True,
    verbose=True,
)

out = run_one_by_global_i(
    cfg,
    global_i=0,
    make_plot=True,
)

out["record"]["plot_path"]

In [ ]:
import pandas as pd
from pathlib import Path

run_dir = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/LSSTMONTS_xi_baseline_v5p3p5_hiddenParallax_FSPLparallax_fitFSPLNoPiE_t0pm60_DetectionFlag"
)

summary = pd.read_parquet(
    run_dir / "logs" / "run_summary.parquet"
)

summary["status"].value_counts(dropna=False)

In [ ]:
summary[
    [
        "global_i",
        "catalog_event_id",
        "status",
        "sim_fit_status",
        "t0_jd",
        "tE_catalog_days",
        "fit_n_points_total",
        "model_dir",
        "fit_dir",
        "results_dir",
        "log_file",
    ]
].head()

In [ ]:
failed = summary[summary["status"] == "failed"].copy()

failed[
    [
        "global_i",
        "catalog_event_id",
        "status",
        "sim_fit_status",
        "error",
        "log_file",
    ]
]

for _, row in failed.head(5).iterrows():
    log_file = Path(row["log_file"])

    print("\n" + "=" * 100)
    print("global_i =", row["global_i"])
    print("catalog_event_id =", row["catalog_event_id"])
    print("error =", row.get("error", ""))
    print("log_file =", log_file)
    print("=" * 100)

    if log_file.exists():
        text = log_file.read_text(errors="replace")
        print(text[-6000:])
    else:
        print("No existe el log")

In [ ]:
import pandas as pd
from pathlib import Path

run_dir = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/"
    "LSSTMONTS_xi_baseline_v5p3p5_hiddenParallax_FSPLparallax_fitFSPLNoPiE_t0pm60_DetectionFlag"
)

prepared = pd.read_parquet(
    run_dir / "catalogs" / "lsstmonts_prepared.parquet"
)

bands = ["u", "g", "r", "i", "z", "y"]

for b in bands:
    blend_col = f"blend_{b}"
    flag_col = f"DetectionFlag_{b}"

    if flag_col not in prepared.columns:
        print(b, "no tiene", flag_col)
        continue

    print("\n", "=" * 80)
    print("band =", b)
    print(
        pd.crosstab(
            prepared[blend_col] == 0,
            prepared[flag_col] == 0,
            rownames=[f"{blend_col} == 0"],
            colnames=[f"{flag_col} == 0"],
        )
    )

In [ ]:
cols = [
    "global_i",
    "catalog_event_id",
    "status",
    "sim_fit_status",
    "t0_jd",
    "t0_reference_jd",
    "t0_reference_raw_first_maf_jd",
    "t0_reference_first_filter",
    "t0_reference_visible_bands",
    "event_first_jd_after_filters",
    "event_first_band_after_filters",
    "event_first_minus_t0_reference_days",
    "fit_n_points_total",
]

summary[cols].head(20)

In [ ]:
summary["t0_reference_check_status"].value_counts(dropna=False)

In [ ]:
import numpy as np

summary["dt0_fit_true"] = summary["t0_fit"] - summary["t0_jd"]

summary[
    np.isclose(np.abs(summary["dt0_fit_true"]), 60.0, atol=1e-2)
][
    [
        "global_i",
        "catalog_event_id",
        "t0_jd",
        "t0_fit",
        "dt0_fit_true",
        "tE_catalog_days",
        "chichi",
        "fit_n_points_total",
    ]
]

In [ ]:
import pandas as pd
from pathlib import Path

row = summary.loc[summary["global_i"] == 16].iloc[0]

true_dir = Path(row["results_dir"]) / "true"
true_files = sorted(true_dir.glob("true_rr_*.parquet"))

print(true_files)

true = pd.read_parquet(true_files[0])

phot_cols = [c for c in true.columns if c.startswith("phot_")]

true[phot_cols].T

In [ ]:
row = summary.loc[summary["global_i"] == 16].iloc[0]

for c in [
    "t0_reference_jd",
    "event_first_jd_after_filters",
    "event_first_minus_t0_reference_days",
    "t0_reference_first_filter",
    "event_first_band_after_filters",
    "t0_reference_raw_first_maf_jd",
    "t0_reference_visible_bands",
]:
    print(c, repr(row[c]))

In [ ]:
delta_days = float(row["event_first_minus_t0_reference_days"])
print(delta_days, "days")
print(delta_days * 24 * 3600, "seconds")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Make preliminary figures for the LSSTMONTS hidden-parallax run.

Figures produced:
    1. rho_fit vs rho_true
    2. relative rho error distribution
    3. Delta chi2 or reduced chi2 diagnostic
    4. table of representative events to plot individually

This script works with either:
    runs/<run_name>/logs/run_summary.parquet
or chunked outputs:
    runs/<run_name>/chunk_*/logs/run_summary.parquet
"""

from pathlib import Path
import argparse
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# User defaults
# ============================================================

DEFAULT_RUN_DIR = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/"
    "LSSTMONTS_xi_baseline_v5p3p5_hiddenParallax_FSPLparallax_fitFSPLNoPiE_t0pm60_DetectionFlag"
)

DEFAULT_CONFIG = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/"
    "lsstmonts_catalog_sedighe/config_lsstmonts_baseline_v5p3p5.json"
)


# ============================================================
# Helpers
# ============================================================

def load_run_name_from_config(config_path):
    config_path = Path(config_path)

    if not config_path.exists():
        return None

    with open(config_path, "r") as f:
        cfg = json.load(f)

    return cfg.get("run_name", None)


def infer_run_dir_from_config(config_path):
    config_path = Path(config_path)

    with open(config_path, "r") as f:
        cfg = json.load(f)

    run_name = cfg["run_name"]
    path_storage = cfg.get(
        "path_storage",
        "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs",
    )

    return Path(path_storage) / run_name


def find_summary_files(run_dir):
    run_dir = Path(run_dir)

    files = []

    direct = run_dir / "logs" / "run_summary.parquet"
    if direct.exists():
        files.append(direct)

    files.extend(sorted(run_dir.glob("chunk_*/logs/run_summary.parquet")))

    if len(files) == 0:
        raise FileNotFoundError(
            f"No encontré run_summary.parquet en {run_dir} "
            "ni en chunk_*/logs/."
        )

    return files


def load_summary(run_dir):
    files = find_summary_files(run_dir)

    tables = []
    for file in files:
        table = pd.read_parquet(file)
        table["summary_file"] = str(file)
        table["chunk_dir"] = str(file.parents[1])
        tables.append(table)

    summary = pd.concat(tables, ignore_index=True)

    if "global_i" in summary.columns:
        summary = (
            summary.sort_values("global_i")
            .drop_duplicates("global_i", keep="last")
            .reset_index(drop=True)
        )

    return summary


def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def find_first_file(directory, patterns):
    directory = Path(directory)

    for pattern in patterns:
        files = sorted(directory.rglob(pattern))
        if len(files) > 0:
            return files[0]

    return None


def read_one_row_parquet(file):
    if file is None:
        return {}

    try:
        df = pd.read_parquet(file)
    except Exception:
        return {}

    if len(df) == 0:
        return {}

    row = df.iloc[0].to_dict()
    return row


def read_fit_params_from_results_dir(results_dir):
    """
    Reads fitted parameters from the per-event fit_rr parquet.

    Expected usual location:
        results_dir/fit_rr/fit_rr_*.parquet

    The function is intentionally permissive because different versions of
    the pipeline may use slightly different filenames.
    """

    results_dir = Path(results_dir)

    fit_file = find_first_file(
        results_dir,
        [
            "fit_rr_*.parquet",
            "*fit_rr*.parquet",
            "*fit*.parquet",
        ],
    )

    row = read_one_row_parquet(fit_file)

    out = {
        "fit_file": str(fit_file) if fit_file is not None else "",
        "rho_fit": np.nan,
        "rho_fit_err": np.nan,
        "t0_fit": np.nan,
        "u0_fit": np.nan,
        "tE_fit": np.nan,
        "chi2_fit": np.nan,
        "dof_fit": np.nan,
    }

    aliases = {
        "rho_fit": ["rho", "rho_fit", "fit_rho"],
        "rho_fit_err": ["rho_err", "rho_fit_err", "fit_rho_err"],
        "t0_fit": ["t0", "t0_fit", "fit_t0"],
        "u0_fit": ["u0", "u0_fit", "fit_u0"],
        "tE_fit": ["tE", "tE_fit", "fit_tE"],
        "chi2_fit": ["chichi", "chi2", "chi2_fit", "fit_chi2"],
        "dof_fit": ["dof", "dof_fit", "fit_dof"],
    }

    for key, names in aliases.items():
        for name in names:
            if name in row:
                try:
                    out[key] = float(row[name])
                except Exception:
                    out[key] = np.nan
                break

    return out


def attach_fit_params(summary, max_events=None):
    """
    Add fit parameters to the summary table by reading each event's fit_rr file.

    For a preliminary sample, use max_events=10000.
    """

    if "results_dir" not in summary.columns:
        raise KeyError(
            "summary no tiene la columna results_dir. "
            "No puedo localizar los fit_rr parquets."
        )

    if max_events is not None:
        table = summary.head(int(max_events)).copy()
    else:
        table = summary.copy()

    fit_rows = []

    for k, (_, row) in enumerate(table.iterrows(), start=1):
        fit_rows.append(
            read_fit_params_from_results_dir(row["results_dir"])
        )

        if k == 1 or k % 500 == 0 or k == len(table):
            print(f"[fit params] {k}/{len(table)}")

    fit_table = pd.DataFrame(fit_rows)

    out = pd.concat(
        [
            table.reset_index(drop=True),
            fit_table.reset_index(drop=True),
        ],
        axis=1,
    )

    return out


def prepare_metrics_table(summary_fit):
    df = summary_fit.copy()

    rho_true_col = first_existing_column(
        df,
        ["rho_catalog", "rho_true", "rho"],
    )

    if rho_true_col is None:
        raise KeyError(
            "No encontré columna de rho verdadero. "
            "Busqué rho_catalog, rho_true, rho."
        )

    df["rho_true_for_plot"] = pd.to_numeric(
        df[rho_true_col],
        errors="coerce",
    )

    df["rho_fit_for_plot"] = pd.to_numeric(
        df["rho_fit"],
        errors="coerce",
    )

    df["rho_relative_error"] = (
        df["rho_fit_for_plot"] - df["rho_true_for_plot"]
    ) / df["rho_true_for_plot"]

    df["rho_log10_ratio"] = np.log10(
        df["rho_fit_for_plot"] / df["rho_true_for_plot"]
    )

    if "chi2_fit" in df.columns and "dof_fit" in df.columns:
        df["chi2_red_fit"] = df["chi2_fit"] / df["dof_fit"]
    else:
        df["chi2_red_fit"] = np.nan

    # Keep only successful finite rho measurements.
    good = np.isfinite(df["rho_true_for_plot"])
    good &= np.isfinite(df["rho_fit_for_plot"])
    good &= df["rho_true_for_plot"] > 0.0
    good &= df["rho_fit_for_plot"] > 0.0

    if "status" in df.columns:
        good &= df["status"].astype(str).isin(["ok"])

    df = df.loc[good].copy().reset_index(drop=True)

    return df


# ============================================================
# Figures
# ============================================================

def plot_rho_recovery(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    x = df["rho_true_for_plot"].to_numpy(dtype=float)
    y = df["rho_fit_for_plot"].to_numpy(dtype=float)

    lo = np.nanmin([np.nanmin(x), np.nanmin(y)])
    hi = np.nanmax([np.nanmax(x), np.nanmax(y)])

    fig, ax = plt.subplots(figsize=(6.5, 5.5))

    ax.scatter(x, y, s=8, alpha=0.35)

    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.5)

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"Injected $\rho_{\rm true}$")
    ax.set_ylabel(r"Recovered $\rho_{\rm fit}$")
    ax.set_title(r"Finite-source parameter recovery")

    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / "rho_fit_vs_rho_true.pdf")
    fig.savefig(output_dir / "rho_fit_vs_rho_true.png", dpi=250)

    plt.close(fig)


def plot_rho_relative_error(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    values = df["rho_relative_error"].to_numpy(dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        print("[warning] No finite rho_relative_error values.")
        return

    # Robust plotting limits to avoid one outlier compressing the whole plot.
    q1, q99 = np.nanpercentile(values, [1, 99])
    values_plot = values[(values >= q1) & (values <= q99)]

    fig, ax = plt.subplots(figsize=(6.5, 5.0))

    ax.hist(values_plot, bins=60, histtype="step", linewidth=1.5)

    ax.axvline(0.0, linestyle="--", linewidth=1.2)

    ax.set_xlabel(
        r"$(\rho_{\rm fit}-\rho_{\rm true})/\rho_{\rm true}$"
    )
    ax.set_ylabel("Number of events")
    ax.set_title(r"Relative error in recovered $\rho$")

    ax.grid(True, alpha=0.3)

    text = (
        rf"$N={len(values)}$" "\n"
        rf"median $={np.nanmedian(values):.3g}$" "\n"
        rf"16--84\% $=({np.nanpercentile(values,16):.3g},"
        rf"{np.nanpercentile(values,84):.3g})$"
    )

    ax.text(
        0.03,
        0.97,
        text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    )

    fig.tight_layout()

    fig.savefig(output_dir / "rho_relative_error_distribution.pdf")
    fig.savefig(output_dir / "rho_relative_error_distribution.png", dpi=250)

    plt.close(fig)


def plot_delta_chi2_or_chi2(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    delta_col = first_existing_column(
        df,
        [
            "delta_chi2_catalog",
            "Delta_chi2",
            "delta_chi2",
            "deltachi2",
            "DeltaChi2",
        ],
    )

    if delta_col is not None:
        values = pd.to_numeric(df[delta_col], errors="coerce").to_numpy(float)
        values = values[np.isfinite(values) & (values > 0.0)]

        if len(values) == 0:
            print(f"[warning] {delta_col} exists but has no positive finite values.")
            return

        fig, ax = plt.subplots(figsize=(6.5, 5.0))

        bins = np.logspace(
            np.log10(np.nanmin(values)),
            np.log10(np.nanmax(values)),
            60,
        )

        ax.hist(values, bins=bins, histtype="step", linewidth=1.5)
        ax.set_xscale("log")

        ax.set_xlabel(r"$\Delta\chi^2$")
        ax.set_ylabel("Number of events")
        ax.set_title(r"Catalog $\Delta\chi^2$ diagnostic")

        ax.grid(True, alpha=0.3)

        fig.tight_layout()

        fig.savefig(output_dir / "delta_chi2_distribution.pdf")
        fig.savefig(output_dir / "delta_chi2_distribution.png", dpi=250)

        plt.close(fig)

        return

    # Fallback: reduced chi2 from the no-parallax fit.
    values = df["chi2_red_fit"].to_numpy(dtype=float)
    values = values[np.isfinite(values) & (values > 0.0)]

    if len(values) == 0:
        print(
            "[warning] No encontré Delta chi2 ni chi2_red_fit finito. "
            "No hago figura de chi2."
        )
        return

    q1, q99 = np.nanpercentile(values, [1, 99])
    values_plot = values[(values >= q1) & (values <= q99)]

    fig, ax = plt.subplots(figsize=(6.5, 5.0))

    ax.hist(values_plot, bins=60, histtype="step", linewidth=1.5)

    ax.axvline(1.0, linestyle="--", linewidth=1.2)

    ax.set_xlabel(r"$\chi^2/{\rm dof}$")
    ax.set_ylabel("Number of events")
    ax.set_title(r"No-parallax FSPL fit quality")

    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / "reduced_chi2_distribution.pdf")
    fig.savefig(output_dir / "reduced_chi2_distribution.png", dpi=250)

    plt.close(fig)


def select_representative_events(df, output_dir, n_each=3):
    """
    Select events useful for the paper figures.

    The output table gives global_i, catalog_event_id, results_dir and metrics.
    Use these rows later to make detailed light-curve/trajectory plots.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    tmp = df.copy()
    tmp["abs_rho_relative_error"] = np.abs(tmp["rho_relative_error"])

    # Good rho recovery.
    rows.append(
        tmp.sort_values("abs_rho_relative_error")
        .head(n_each)
        .assign(selection_reason="best_rho_recovery")
    )

    # Median-like rho recovery.
    median_value = np.nanmedian(tmp["abs_rho_relative_error"])
    tmp["distance_to_median_abs_error"] = np.abs(
        tmp["abs_rho_relative_error"] - median_value
    )
    rows.append(
        tmp.sort_values("distance_to_median_abs_error")
        .head(n_each)
        .assign(selection_reason="typical_rho_recovery")
    )

    # Worst rho recovery.
    rows.append(
        tmp.sort_values("abs_rho_relative_error", ascending=False)
        .head(n_each)
        .assign(selection_reason="largest_rho_bias")
    )

    if "chi2_red_fit" in tmp.columns:
        chi = tmp[np.isfinite(tmp["chi2_red_fit"])].copy()
        if len(chi) > 0:
            rows.append(
                chi.sort_values("chi2_red_fit", ascending=False)
                .head(n_each)
                .assign(selection_reason="largest_reduced_chi2")
            )

    selected = pd.concat(rows, ignore_index=True)

    keep_cols = [
        "selection_reason",
        "global_i",
        "catalog_row",
        "catalog_event_id",
        "status",
        "rho_true_for_plot",
        "rho_fit_for_plot",
        "rho_relative_error",
        "rho_log10_ratio",
        "chi2_fit",
        "dof_fit",
        "chi2_red_fit",
        "t0_jd",
        "tE_catalog_days",
        "u0",
        "piE",
        "xi_deg",
        "catalog_available_bands",
        "results_dir",
        "fit_file",
    ]

    keep_cols = [c for c in keep_cols if c in selected.columns]

    selected = selected[keep_cols].drop_duplicates(
        subset=[c for c in ["global_i"] if c in selected.columns],
        keep="first",
    )

    selected.to_csv(
        output_dir / "representative_events_for_lightcurves.csv",
        index=False,
    )

    selected.to_latex(
        output_dir / "representative_events_for_lightcurves.tex",
        index=False,
        escape=False,
        float_format="%.4g",
    )

    return selected


# ============================================================
# Optional: simple event diagnostic panel from summary only
# ============================================================

def plot_event_summary_panel(row, output_dir):
    """
    This does not replace the full light-curve + trajectory figure.
    It is a compact diagnostic panel for one event using summary/fit metadata.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    event_id = int(row["global_i"])

    labels = []
    values = []

    for label, col in [
        (r"$\rho_{\rm true}$", "rho_true_for_plot"),
        (r"$\rho_{\rm fit}$", "rho_fit_for_plot"),
        (r"$\Delta\rho/\rho$", "rho_relative_error"),
        (r"$t_E$ [days]", "tE_catalog_days"),
        (r"$u_0$", "u0"),
        (r"$\pi_E$", "piE"),
        (r"$\xi$ [deg]", "xi_deg"),
        (r"$\chi^2/{\rm dof}$", "chi2_red_fit"),
    ]:
        if col in row.index and np.isfinite(row[col]):
            labels.append(label)
            values.append(float(row[col]))

    fig, ax = plt.subplots(figsize=(6.5, 4.5))

    y = np.arange(len(values))
    ax.barh(y, values)

    ax.set_yticks(y)
    ax.set_yticklabels(labels)

    ax.set_xlabel("Value")
    ax.set_title(f"Event {event_id}: metadata diagnostic")

    ax.grid(True, axis="x", alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / f"event_{event_id:07d}_summary_panel.pdf")
    fig.savefig(output_dir / f"event_{event_id:07d}_summary_panel.png", dpi=250)

    plt.close(fig)


# ============================================================
# Main
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--run-dir",
        default=None,
        help="Run directory. If omitted, inferred from --config.",
    )

    parser.add_argument(
        "--config",
        default=str(DEFAULT_CONFIG),
        help="Config file used by the runner.",
    )

    parser.add_argument(
        "--max-events",
        type=int,
        default=10000,
        help="Maximum number of successful events used for preliminary figures.",
    )

    parser.add_argument(
        "--output-dir",
        default=None,
        help="Directory where figures are written.",
    )

    parser.add_argument(
        "--make-event-summary-panels",
        action="store_true",
        help="Make simple per-event metadata panels for selected events.",
    )

    args = parser.parse_args()

    if args.run_dir is None:
        run_dir = infer_run_dir_from_config(args.config)
    else:
        run_dir = Path(args.run_dir)

    if args.output_dir is None:
        output_dir = run_dir / "figures" / f"sample_{args.max_events}"
    else:
        output_dir = Path(args.output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    print("run_dir    =", run_dir)
    print("output_dir =", output_dir)

    print("[1] Loading summary...")
    summary = load_summary(run_dir)

    print("Total rows in summary:", len(summary))
    if "status" in summary.columns:
        print(summary["status"].value_counts(dropna=False))

    if "status" in summary.columns:
        ok_summary = summary[summary["status"].astype(str) == "ok"].copy()
    else:
        ok_summary = summary.copy()

    ok_summary = ok_summary.sort_values(
        "global_i" if "global_i" in ok_summary.columns else ok_summary.index.name
    ).reset_index(drop=True)

    if args.max_events is not None:
        ok_summary = ok_summary.head(args.max_events).copy()

    print("Events used for preliminary figures:", len(ok_summary))

    print("[2] Reading fit parameters...")
    summary_fit = attach_fit_params(
        ok_summary,
        max_events=None,
    )

    summary_fit.to_parquet(
        output_dir / "sample_with_fit_params.parquet",
        index=False,
    )

    summary_fit.to_csv(
        output_dir / "sample_with_fit_params.csv",
        index=False,
    )

    print("[3] Preparing metrics...")
    metrics = prepare_metrics_table(summary_fit)

    metrics.to_parquet(
        output_dir / "sample_metrics.parquet",
        index=False,
    )

    metrics.to_csv(
        output_dir / "sample_metrics.csv",
        index=False,
    )

    print("Events with finite rho_true and rho_fit:", len(metrics))

    print("[4] Making figures...")
    plot_rho_recovery(metrics, output_dir)
    plot_rho_relative_error(metrics, output_dir)
    plot_delta_chi2_or_chi2(metrics, output_dir)

    print("[5] Selecting representative events...")
    selected = select_representative_events(metrics, output_dir)

    print(selected)

    if args.make_event_summary_panels:
        for _, row in selected.iterrows():
            plot_event_summary_panel(row, output_dir)

    print("Done.")
    print("Figures written to:")
    print(output_dir)


if __name__ == "__main__":
    main() 

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Make figures showing events where FSPL+parallax light curves are confused
with FSPL no-parallax fits.

The true injected model is assumed to be:
    FSPL + annual parallax

The fitted model is assumed to be:
    FSPL without parallax

We identify "confused" events as cases where the no-parallax FSPL fit is good
and the recovered rho is acceptable.

Default criteria:
    chi2_red_fit < 1.5
    |rho_fit - rho_true| / rho_true < 0.5
    sigma_rho / rho_fit < 0.5, if available
"""

from pathlib import Path
import argparse
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Defaults
# ============================================================

DEFAULT_CONFIG = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/"
    "lsstmonts_catalog_sedighe/config_lsstmonts_baseline_v5p3p5.json"
)


# ============================================================
# I/O helpers
# ============================================================

def infer_run_dir_from_config(config_path):
    config_path = Path(config_path)

    with open(config_path, "r") as f:
        cfg = json.load(f)

    run_name = cfg["run_name"]
    path_storage = cfg.get(
        "path_storage",
        "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs",
    )

    return Path(path_storage) / run_name


def find_summary_files(run_dir):
    run_dir = Path(run_dir)

    files = []

    direct = run_dir / "logs" / "run_summary.parquet"
    if direct.exists():
        files.append(direct)

    files.extend(sorted(run_dir.glob("chunk_*/logs/run_summary.parquet")))

    if len(files) == 0:
        raise FileNotFoundError(
            f"No encontré run_summary.parquet en {run_dir} "
            "ni en chunk_*/logs/."
        )

    return files


def load_summary(run_dir):
    files = find_summary_files(run_dir)

    tables = []

    for file in files:
        table = pd.read_parquet(file)
        table["summary_file"] = str(file)
        table["chunk_dir"] = str(file.parents[1])
        tables.append(table)

    summary = pd.concat(tables, ignore_index=True)

    if "global_i" in summary.columns:
        summary = (
            summary.sort_values("global_i")
            .drop_duplicates("global_i", keep="last")
            .reset_index(drop=True)
        )

    return summary


def find_first_file(directory, patterns):
    directory = Path(directory)

    for pattern in patterns:
        files = sorted(directory.rglob(pattern))
        if len(files) > 0:
            return files[0]

    return None


def read_one_row_parquet(file):
    if file is None:
        return {}

    try:
        df = pd.read_parquet(file)
    except Exception:
        return {}

    if len(df) == 0:
        return {}

    return df.iloc[0].to_dict()


def read_fit_params_from_results_dir(results_dir):
    """
    Read fitted parameters from each event output.

    It first searches for fit_rr parquets, then for any fit-like parquet.
    """

    results_dir = Path(results_dir)

    fit_file = find_first_file(
        results_dir,
        [
            "fit_rr_*.parquet",
            "*fit_rr*.parquet",
            "*fit*.parquet",
        ],
    )

    row = read_one_row_parquet(fit_file)

    out = {
        "fit_file": str(fit_file) if fit_file is not None else "",
        "rho_fit_file": np.nan,
        "rho_err_file": np.nan,
        "t0_fit_file": np.nan,
        "u0_fit_file": np.nan,
        "tE_fit_file": np.nan,
        "chi2_fit_file": np.nan,
        "dof_fit_file": np.nan,
    }

    aliases = {
        "rho_fit_file": ["rho", "rho_fit", "fit_rho"],
        "rho_err_file": ["rho_err", "rho_fit_err", "fit_rho_err"],
        "t0_fit_file": ["t0", "t0_fit", "fit_t0"],
        "u0_fit_file": ["u0", "u0_fit", "fit_u0"],
        "tE_fit_file": ["tE", "tE_fit", "fit_tE"],
        "chi2_fit_file": ["chichi", "chi2", "chi2_fit", "fit_chi2"],
        "dof_fit_file": ["dof", "dof_fit", "fit_dof"],
    }

    for key, names in aliases.items():
        for name in names:
            if name in row:
                try:
                    out[key] = float(row[name])
                except Exception:
                    out[key] = np.nan
                break

    return out


def attach_fit_params(summary, max_events=None):
    """
    Attach fit_rr parameters to the summary table.

    If your runner already saved covariance quantities into summary,
    they are preserved.
    """

    if "results_dir" not in summary.columns:
        raise KeyError(
            "summary no tiene results_dir; no puedo encontrar fit_rr."
        )

    if max_events is not None:
        table = summary.head(int(max_events)).copy()
    else:
        table = summary.copy()

    rows = []

    for k, (_, row) in enumerate(table.iterrows(), start=1):
        rows.append(read_fit_params_from_results_dir(row["results_dir"]))

        if k == 1 or k % 500 == 0 or k == len(table):
            print(f"[fit params] {k}/{len(table)}")

    fit_table = pd.DataFrame(rows)

    return pd.concat(
        [table.reset_index(drop=True), fit_table.reset_index(drop=True)],
        axis=1,
    )


def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


# ============================================================
# Metrics
# ============================================================

def build_metrics(summary_fit):
    df = summary_fit.copy()

    # True rho.
    rho_true_col = first_existing_column(
        df,
        ["rho_catalog", "rho_true", "rho"],
    )

    if rho_true_col is None:
        raise KeyError(
            "No encontré rho verdadero. Busqué rho_catalog, rho_true, rho."
        )

    df["rho_true"] = pd.to_numeric(df[rho_true_col], errors="coerce")

    # Fitted rho: prefer covariance-runner value if available, otherwise file.
    rho_fit_col = first_existing_column(
        df,
        [
            "rho_fit_from_best_model",
            "rho_fit",
            "rho_fit_file",
        ],
    )

    if rho_fit_col is None:
        raise KeyError(
            "No encontré rho fit. Busqué rho_fit_from_best_model, "
            "rho_fit, rho_fit_file."
        )

    df["rho_fit"] = pd.to_numeric(df[rho_fit_col], errors="coerce")

    # Chi2 and dof.
    chi2_col = first_existing_column(
        df,
        ["chi2_fit", "chi2_fit_file", "chichi"],
    )

    dof_col = first_existing_column(
        df,
        ["dof_fit", "dof_fit_file", "dof"],
    )

    if chi2_col is not None and dof_col is not None:
        df["chi2_fit_used"] = pd.to_numeric(df[chi2_col], errors="coerce")
        df["dof_fit_used"] = pd.to_numeric(df[dof_col], errors="coerce")
        df["chi2_red_fit"] = df["chi2_fit_used"] / df["dof_fit_used"]
    elif "chi2_red_fit" in df.columns:
        df["chi2_red_fit"] = pd.to_numeric(
            df["chi2_red_fit"],
            errors="coerce",
        )
    else:
        df["chi2_red_fit"] = np.nan

    # sigma_rho/rho.
    if "sigma_rho_over_rho_fit" in df.columns:
        df["sigma_rho_over_rho"] = pd.to_numeric(
            df["sigma_rho_over_rho_fit"],
            errors="coerce",
        )
    elif "rho_err_cov" in df.columns:
        df["sigma_rho_over_rho"] = (
            pd.to_numeric(df["rho_err_cov"], errors="coerce")
            / df["rho_fit"]
        )
    elif "rho_err_file" in df.columns:
        df["sigma_rho_over_rho"] = (
            pd.to_numeric(df["rho_err_file"], errors="coerce")
            / df["rho_fit"]
        )
    else:
        df["sigma_rho_over_rho"] = np.nan

    # Basic derived quantities.
    df["rho_relative_error"] = (
        df["rho_fit"] - df["rho_true"]
    ) / df["rho_true"]

    df["abs_rho_relative_error"] = np.abs(df["rho_relative_error"])

    df["rho_log10_ratio"] = np.log10(df["rho_fit"] / df["rho_true"])

    if "piE" in df.columns:
        df["piE_used"] = pd.to_numeric(df["piE"], errors="coerce")
    else:
        df["piE_used"] = np.nan

    if "tE_catalog_days" in df.columns:
        df["tE_used"] = pd.to_numeric(
            df["tE_catalog_days"],
            errors="coerce",
        )
    elif "tE_true" in df.columns:
        df["tE_used"] = pd.to_numeric(df["tE_true"], errors="coerce")
    else:
        df["tE_used"] = np.nan

    # Keep only events where comparison makes sense.
    good = np.isfinite(df["rho_true"])
    good &= np.isfinite(df["rho_fit"])
    good &= df["rho_true"] > 0.0
    good &= df["rho_fit"] > 0.0

    if "status" in df.columns:
        good &= df["status"].astype(str).eq("ok")

    df = df.loc[good].copy().reset_index(drop=True)

    return df


def classify_confused_events(
    df,
    chi2_red_max=1.5,
    rho_rel_error_max=0.5,
    sigma_rho_over_rho_max=0.5,
    require_sigma=False,
):
    df = df.copy()

    confused = np.ones(len(df), dtype=bool)

    confused &= np.isfinite(df["chi2_red_fit"])
    confused &= df["chi2_red_fit"] < float(chi2_red_max)

    confused &= np.isfinite(df["abs_rho_relative_error"])
    confused &= df["abs_rho_relative_error"] < float(rho_rel_error_max)

    sigma = df["sigma_rho_over_rho"].to_numpy(float)

    if require_sigma:
        confused &= np.isfinite(sigma)
        confused &= sigma < float(sigma_rho_over_rho_max)
    else:
        sigma_ok_or_missing = (
            ~np.isfinite(sigma)
            | (sigma < float(sigma_rho_over_rho_max))
        )
        confused &= sigma_ok_or_missing

    df["confused_fspl_noparallax"] = confused

    return df


# ============================================================
# Figures
# ============================================================

def plot_rho_recovery_confused(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_df = df.copy()
    conf = df[df["confused_fspl_noparallax"]].copy()

    x = all_df["rho_true"].to_numpy(float)
    y = all_df["rho_fit"].to_numpy(float)

    lo = np.nanmin([np.nanmin(x), np.nanmin(y)])
    hi = np.nanmax([np.nanmax(x), np.nanmax(y)])

    fig, ax = plt.subplots(figsize=(6.6, 5.6))

    ax.scatter(
        all_df["rho_true"],
        all_df["rho_fit"],
        s=8,
        alpha=0.18,
        label="All fitted events",
    )

    ax.scatter(
        conf["rho_true"],
        conf["rho_fit"],
        s=16,
        alpha=0.75,
        label="Confused with FSPL no parallax",
    )

    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.2)

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"Injected $\rho_{\rm true}$")
    ax.set_ylabel(r"Recovered $\rho_{\rm fit}$")
    ax.set_title(r"Events absorbed by a no-parallax FSPL model")

    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / "confused_rho_fit_vs_rho_true.pdf")
    fig.savefig(output_dir / "confused_rho_fit_vs_rho_true.png", dpi=250)

    plt.close(fig)


def plot_chi2_vs_rho_error(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_df = df.copy()
    conf = df[df["confused_fspl_noparallax"]].copy()

    fig, ax = plt.subplots(figsize=(6.8, 5.4))

    ax.scatter(
        all_df["chi2_red_fit"],
        all_df["abs_rho_relative_error"],
        s=8,
        alpha=0.18,
        label="All fitted events",
    )

    ax.scatter(
        conf["chi2_red_fit"],
        conf["abs_rho_relative_error"],
        s=16,
        alpha=0.75,
        label="Confused",
    )

    ax.axvline(1.5, linestyle="--", linewidth=1.1)
    ax.axhline(0.5, linestyle="--", linewidth=1.1)

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"$\chi^2/{\rm dof}$ of FSPL no-parallax fit")
    ax.set_ylabel(r"$|\rho_{\rm fit}-\rho_{\rm true}|/\rho_{\rm true}$")
    ax.set_title(r"Good no-parallax fits with stable $\rho$ recovery")

    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / "confused_chi2red_vs_rho_error.pdf")
    fig.savefig(output_dir / "confused_chi2red_vs_rho_error.png", dpi=250)

    plt.close(fig)


def plot_piE_tE_confused(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    good = np.isfinite(df["piE_used"]) & np.isfinite(df["tE_used"])
    good &= df["piE_used"] > 0.0
    good &= df["tE_used"] > 0.0

    sub = df.loc[good].copy()

    if len(sub) == 0:
        print("[warning] No hay piE/tE finitos para esta figura.")
        return

    conf = sub[sub["confused_fspl_noparallax"]].copy()

    fig, ax = plt.subplots(figsize=(6.8, 5.4))

    ax.scatter(
        sub["tE_used"],
        sub["piE_used"],
        s=8,
        alpha=0.18,
        label="All fitted events",
    )

    ax.scatter(
        conf["tE_used"],
        conf["piE_used"],
        s=16,
        alpha=0.75,
        label="Confused",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"$t_E$ [days]")
    ax.set_ylabel(r"Injected $\pi_E$")
    ax.set_title(r"Where parallax is hidden by FSPL no-parallax fits")

    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / "confused_piE_vs_tE.pdf")
    fig.savefig(output_dir / "confused_piE_vs_tE.png", dpi=250)

    plt.close(fig)


def plot_sigma_vs_rho_error(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    good = np.isfinite(df["sigma_rho_over_rho"])
    good &= df["sigma_rho_over_rho"] > 0.0
    good &= np.isfinite(df["abs_rho_relative_error"])
    good &= df["abs_rho_relative_error"] > 0.0

    sub = df.loc[good].copy()

    if len(sub) == 0:
        print("[warning] No hay sigma_rho/rho para esta figura.")
        return

    conf = sub[sub["confused_fspl_noparallax"]].copy()

    fig, ax = plt.subplots(figsize=(6.8, 5.4))

    ax.scatter(
        sub["sigma_rho_over_rho"],
        sub["abs_rho_relative_error"],
        s=8,
        alpha=0.18,
        label="All fitted events",
    )

    ax.scatter(
        conf["sigma_rho_over_rho"],
        conf["abs_rho_relative_error"],
        s=16,
        alpha=0.75,
        label="Confused",
    )

    ax.axvline(0.5, linestyle="--", linewidth=1.1)
    ax.axhline(0.5, linestyle="--", linewidth=1.1)

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"$\sigma_\rho/\rho_{\rm fit}$")
    ax.set_ylabel(r"$|\rho_{\rm fit}-\rho_{\rm true}|/\rho_{\rm true}$")
    ax.set_title(r"Measured $\rho$ among confused events")

    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / "confused_sigma_rho_vs_rho_error.pdf")
    fig.savefig(output_dir / "confused_sigma_rho_vs_rho_error.png", dpi=250)

    plt.close(fig)


def plot_confused_fraction_summary(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    n_all = len(df)
    n_conf = int(df["confused_fspl_noparallax"].sum())
    n_not = n_all - n_conf

    fig, ax = plt.subplots(figsize=(5.8, 4.6))

    ax.bar(
        ["Confused", "Not confused"],
        [n_conf, n_not],
    )

    ax.set_ylabel("Number of events")
    ax.set_title("FSPL no-parallax confusion rate")

    frac = n_conf / n_all if n_all > 0 else np.nan
    text = (
        rf"$N={n_all}$" "\n"
        rf"$f_{{\rm confused}}={frac:.3f}$"
    )

    ax.text(
        0.03,
        0.95,
        text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    ax.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()

    fig.savefig(output_dir / "confused_fraction_summary.pdf")
    fig.savefig(output_dir / "confused_fraction_summary.png", dpi=250)

    plt.close(fig)


def save_confused_tables(df, output_dir, n_examples=20):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df.to_parquet(output_dir / "confusion_metrics_all.parquet", index=False)
    df.to_csv(output_dir / "confusion_metrics_all.csv", index=False)

    confused = df[df["confused_fspl_noparallax"]].copy()

    confused.to_parquet(
        output_dir / "confused_fspl_noparallax_events.parquet",
        index=False,
    )
    confused.to_csv(
        output_dir / "confused_fspl_noparallax_events.csv",
        index=False,
    )

    # Representative confused events:
    # prioritize good chi2 and small rho bias, but with nonzero piE.
    selected = confused.copy()

    selected["selection_score"] = (
        np.abs(np.log10(selected["chi2_red_fit"]))
        + selected["abs_rho_relative_error"]
    )

    if "piE_used" in selected.columns:
        selected = selected.sort_values(
            ["selection_score", "piE_used"],
            ascending=[True, False],
        )
    else:
        selected = selected.sort_values("selection_score")

    keep_cols = [
        "global_i",
        "catalog_row",
        "catalog_event_id",
        "chi2_red_fit",
        "rho_true",
        "rho_fit",
        "rho_relative_error",
        "sigma_rho_over_rho",
        "piE_used",
        "tE_used",
        "u0",
        "xi_deg",
        "catalog_available_bands",
        "results_dir",
        "fit_file",
    ]

    keep_cols = [c for c in keep_cols if c in selected.columns]

    examples = selected.head(int(n_examples))[keep_cols].copy()

    examples.to_csv(
        output_dir / "representative_confused_events.csv",
        index=False,
    )

    examples.to_latex(
        output_dir / "representative_confused_events.tex",
        index=False,
        escape=False,
        float_format="%.4g",
    )

    print("N all fitted events:", len(df))
    print("N confused:", len(confused))
    if len(df) > 0:
        print("Fraction confused:", len(confused) / len(df))
    print("Representative confused events:")
    print(examples)


# ============================================================
# Main
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--config",
        default=str(DEFAULT_CONFIG),
    )

    parser.add_argument(
        "--run-dir",
        default=None,
    )

    parser.add_argument(
        "--max-events",
        type=int,
        default=10000,
    )

    parser.add_argument(
        "--chi2-red-max",
        type=float,
        default=1.5,
    )

    parser.add_argument(
        "--rho-rel-error-max",
        type=float,
        default=0.5,
    )

    parser.add_argument(
        "--sigma-rho-over-rho-max",
        type=float,
        default=0.5,
    )

    parser.add_argument(
        "--require-sigma",
        action="store_true",
        help=(
            "Require finite sigma_rho/rho for classification. "
            "Without this flag, events with missing sigma are allowed."
        ),
    )

    parser.add_argument(
        "--output-dir",
        default=None,
    )

    args = parser.parse_args()

    if args.run_dir is None:
        run_dir = infer_run_dir_from_config(args.config)
    else:
        run_dir = Path(args.run_dir)

    if args.output_dir is None:
        output_dir = run_dir / "figures" / "confused_fspl_noparallax"
    else:
        output_dir = Path(args.output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    print("run_dir    =", run_dir)
    print("output_dir =", output_dir)

    print("[1] Loading summary...")
    summary = load_summary(run_dir)

    print("Summary rows:", len(summary))

    if "status" in summary.columns:
        print(summary["status"].value_counts(dropna=False))

    if "status" in summary.columns:
        summary = summary[summary["status"].astype(str) == "ok"].copy()

    summary = summary.sort_values(
        "global_i" if "global_i" in summary.columns else summary.index.name
    ).reset_index(drop=True)

    if args.max_events is not None:
        summary = summary.head(int(args.max_events)).copy()

    print("Events used:", len(summary))

    print("[2] Reading fit parameters...")
    summary_fit = attach_fit_params(summary)

    print("[3] Building metrics...")
    metrics = build_metrics(summary_fit)

    print("[4] Classifying confused events...")
    metrics = classify_confused_events(
        metrics,
        chi2_red_max=args.chi2_red_max,
        rho_rel_error_max=args.rho_rel_error_max,
        sigma_rho_over_rho_max=args.sigma_rho_over_rho_max,
        require_sigma=args.require_sigma,
    )

    print("[5] Saving tables...")
    save_confused_tables(metrics, output_dir)

    print("[6] Making figures...")
    plot_confused_fraction_summary(metrics, output_dir)
    plot_rho_recovery_confused(metrics, output_dir)
    plot_chi2_vs_rho_error(metrics, output_dir)
    plot_piE_tE_confused(metrics, output_dir)
    plot_sigma_vs_rho_error(metrics, output_dir)

    print("Done.")
    print("Figures written to:")
    print(output_dir)


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

run_dir = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/"
    "LSSTMONTS_xi_baseline_v5p3p5_hiddenParallax_FSPLparallax_fitFSPLNoPiE_t0pm60_DetectionFlag"
)

summary_files = sorted(run_dir.glob("chunk_*/logs/run_summary.parquet"))

if len(summary_files) == 0:
    summary_files = [run_dir / "logs" / "run_summary.parquet"]

summary = pd.concat(
    [pd.read_parquet(f) for f in summary_files if f.exists()],
    ignore_index=True,
)

summary = summary[summary["status"].astype(str) == "ok"].copy()
summary = summary.sort_values("global_i").drop_duplicates("global_i").reset_index(drop=True)

print("N summary =", len(summary))
print(summary["status"].value_counts(dropna=False))


def find_first(path, pattern):
    files = sorted(Path(path).rglob(pattern))
    return files[0] if len(files) else None


def read_one(file):
    if file is None:
        return {}
    try:
        df = pd.read_parquet(file)
        if len(df) == 0:
            return {}
        return df.iloc[0].to_dict()
    except Exception:
        return {}


rows = []

for k, row in summary.iterrows():
    results_dir = Path(row["results_dir"])

    fit_file = find_first(results_dir, "fit_rr_*.parquet")
    true_file = find_first(results_dir, "true_rr_*.parquet")

    fit = read_one(fit_file)
    true = read_one(true_file)

    out = row.to_dict()

    out["fit_file"] = str(fit_file) if fit_file is not None else ""
    out["true_file"] = str(true_file) if true_file is not None else ""

    # Fit quantities
    out["t0_fit"] = fit.get("t0", np.nan)
    out["u0_fit"] = fit.get("u0", np.nan)
    out["tE_fit"] = fit.get("tE", np.nan)
    out["rho_fit"] = fit.get("rho", np.nan)
    out["rho_err"] = fit.get("rho_err", np.nan)
    out["chichi"] = fit.get("chichi", fit.get("chi2", np.nan))
    out["dof"] = fit.get("dof", np.nan)

    # True quantities
    out["rho_true"] = true.get("rho", row.get("rho_catalog", np.nan))
    out["tE_true"] = true.get("tE", row.get("tE_catalog_days", np.nan))
    out["u0_true"] = true.get("u0", row.get("u0", np.nan))
    out["piE_true"] = true.get("piE", row.get("piE", np.nan))

    # Peak counts from true parquet
    peak_cols = [
        c for c in true.keys()
        if isinstance(c, str) and c.endswith("_peak")
    ]

    out["n_peak_total"] = np.nansum([
        true.get(c, 0.0) for c in peak_cols
    ]) if len(peak_cols) else np.nan

    rows.append(out)

    if k == 0 or (k + 1) % 500 == 0 or (k + 1) == len(summary):
        print(f"{k+1}/{len(summary)}")

df = pd.DataFrame(rows)

df["chi2_red"] = df["chichi"] / df["dof"]
df["rho_rel_error"] = (df["rho_fit"] - df["rho_true"]) / df["rho_true"]
df["abs_rho_rel_error"] = np.abs(df["rho_rel_error"])
df["sigma_rho_over_rho"] = df["rho_err"] / df["rho_fit"]

df["at_u0_bound"] = np.isclose(np.abs(df["u0_fit"]), 5.0, rtol=0, atol=1e-2)
df["at_tE_upper_bound"] = np.isclose(df["tE_fit"], 20000.0, rtol=0, atol=1e-2)
df["bad_boundary_fit"] = df["at_u0_bound"] | df["at_tE_upper_bound"]

print("\nBasic diagnostics")
print("-----------------")
print("finite chi2_red:", np.isfinite(df["chi2_red"]).sum())
print("finite rho_fit:", np.isfinite(df["rho_fit"]).sum())
print("finite rho_true:", np.isfinite(df["rho_true"]).sum())
print("finite n_peak_total:", np.isfinite(df["n_peak_total"]).sum())

print("\nchi2_red percentiles")
print(df["chi2_red"].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]))

print("\nn_peak_total")
print(df["n_peak_total"].value_counts(dropna=False).sort_index().head(20))

print("\nBoundary fits")
print(df["bad_boundary_fit"].value_counts(dropna=False))

m0 = np.ones(len(df), dtype=bool)

m1 = m0 & np.isfinite(df["chi2_red"]) & (df["chi2_red"] < 1.5)

m2 = m1 & (~df["bad_boundary_fit"])

m3 = m2 & np.isfinite(df["n_peak_total"]) & (df["n_peak_total"] >= 5)

m4 = m3 & np.isfinite(df["sigma_rho_over_rho"]) & (df["sigma_rho_over_rho"] < 0.5)

m5 = m4 & np.isfinite(df["abs_rho_rel_error"]) & (df["abs_rho_rel_error"] < 0.5)

print("\nSelection cascade")
print("-----------------")
print("ok fits:", len(df))
print("chi2_red < 1.5:", int(m1.sum()))
print("plus not at bounds:", int(m2.sum()))
print("plus N_peak >= 5:", int(m3.sum()))
print("plus sigma_rho/rho < 0.5:", int(m4.sum()))
print("plus |rho_fit-rho_true|/rho_true < 0.5:", int(m5.sum()))

# Good examples for figures:
# I would start with m3, not m5.
# m3 = good no-parallax fit, not at bounds, peak covered.
candidates = df[m3].copy()

candidates = candidates.sort_values(
    ["chi2_red", "bad_boundary_fit", "n_peak_total"],
    ascending=[True, True, False],
)

out = (
    run_dir
    / "figures"
    / "confused_fspl_noparallax"
    / "confused_candidates_from_fitrr_truerr.csv"
)

out.parent.mkdir(parents=True, exist_ok=True)
candidates.to_csv(out, index=False)

print("\nSaved candidates:")
print(out)

cols = [
    "global_i",
    "chi2_red",
    "n_peak_total",
    "rho_true",
    "rho_fit",
    "rho_rel_error",
    "sigma_rho_over_rho",
    "tE_true",
    "tE_fit",
    "u0_true",
    "u0_fit",
    "piE_true",
    "fit_file",
]

cols = [c for c in cols if c in candidates.columns]

print("\nTop candidates:")
print(candidates[cols].head(20))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

run_dir = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/"
    "LSSTMONTS_xi_baseline_v5p3p5_hiddenParallax_FSPLparallax_fitFSPLNoPiE_t0pm60_DetectionFlag"
)

candidates_file = (
    run_dir
    / "figures"
    / "confused_fspl_noparallax"
    / "confused_candidates_from_fitrr_truerr.csv"
)

df = pd.read_csv(candidates_file)

# ------------------------------------------------------------
# Make sure delta chi2 is present
# ------------------------------------------------------------

def read_delta_chi2(fit_file):
    try:
        fit = pd.read_parquet(fit_file)
        if len(fit) == 0:
            return np.nan
        row = fit.iloc[0]
        for col in ["delta_chi2_true", "delta_chi2", "Delta_chi2"]:
            if col in row.index:
                return float(row[col])
    except Exception:
        pass
    return np.nan

if "delta_chi2_true" not in df.columns:
    df["delta_chi2_true"] = df["fit_file"].apply(read_delta_chi2)

# ------------------------------------------------------------
# Derived quantities
# ------------------------------------------------------------

df["rho_ratio"] = df["rho_fit"] / df["rho_true"]
df["abs_log_rho_ratio"] = np.abs(np.log10(df["rho_ratio"]))
df["chi2_score"] = np.abs(np.log10(df["chi2_red"]))

df["tE_ratio"] = df["tE_fit"] / df["tE_true"]

# ------------------------------------------------------------
# Base scientific definition for light-curve confusion
# ------------------------------------------------------------

base = df.copy()

base = base[np.isfinite(base["chi2_red"])]
base = base[base["chi2_red"] < 1.5]

base = base[base["bad_boundary_fit"] == False]

base = base[np.isfinite(base["n_peak_total"])]
base = base[base["n_peak_total"] >= 5]

# Avoid absurd timescale transformations for display figures.
# This is not a physics cut; it is only to choose readable examples.
base = base[np.isfinite(base["tE_ratio"])]
base = base[(base["tE_ratio"] > 0.1) & (base["tE_ratio"] < 10.0)]

print("Base candidates:", len(base))

print("\nDelta chi2 percentiles in base sample:")
print(base["delta_chi2_true"].describe(percentiles=[0.05, 0.1, 0.5, 0.9, 0.95]))

# ------------------------------------------------------------
# Prefer low Delta chi2, but relax automatically if needed
# ------------------------------------------------------------

selected_pool = None

for dchi2_max in [10, 25, 50, 100, 200, np.inf]:
    pool = base[
        np.isfinite(base["delta_chi2_true"])
        & (base["delta_chi2_true"] < dchi2_max)
    ].copy()

    print(f"Delta chi2 < {dchi2_max}: {len(pool)}")

    if len(pool) >= 6:
        selected_pool = pool
        print("Using Delta chi2 threshold:", dchi2_max)
        break

if selected_pool is None or len(selected_pool) == 0:
    selected_pool = base.copy()
    print("WARNING: using base sample without Delta chi2 cut")

# ------------------------------------------------------------
# Select diverse examples
# ------------------------------------------------------------

selected_rows = []

def add_one(label, table, sort_cols, ascending=True):
    global selected_rows

    if len(table) == 0:
        return

    if isinstance(sort_cols, str):
        sort_cols = [sort_cols]

    ordered = table.sort_values(sort_cols, ascending=ascending)

    used = set()
    if len(selected_rows) > 0:
        used = set(pd.concat(selected_rows)["global_i"].astype(int))

    ordered = ordered[~ordered["global_i"].astype(int).isin(used)]

    if len(ordered) == 0:
        return

    row = ordered.head(1).copy()
    row["selection_reason"] = label
    selected_rows.append(row)

# 1. Best no-parallax confusion: low chi2 and low delta chi2
add_one(
    "low_chi2_low_delta_chi2",
    selected_pool,
    ["delta_chi2_true", "chi2_score"],
    ascending=True,
)

# 2. Many peak points
add_one(
    "well_sampled_peak",
    selected_pool,
    ["n_peak_total", "delta_chi2_true"],
    ascending=[False, True],
)

# 3. Large injected parallax but still fit by no-parallax FSPL
add_one(
    "large_piE_hidden",
    selected_pool[np.isfinite(selected_pool["piE_true"])],
    ["piE_true", "delta_chi2_true"],
    ascending=[False, True],
)

# 4. rho approximately recovered, even if uncertainty is large
add_one(
    "rho_approximately_recovered",
    selected_pool[np.isfinite(selected_pool["rho_rel_error"])],
    ["abs_rho_rel_error", "delta_chi2_true"],
    ascending=True,
)

# 5. rho suppressed by the no-parallax model
add_one(
    "rho_suppressed",
    selected_pool[selected_pool["rho_ratio"] < 0.3],
    ["delta_chi2_true", "chi2_score"],
    ascending=True,
)

# 6. rho inflated by the no-parallax model
add_one(
    "rho_inflated",
    selected_pool[selected_pool["rho_ratio"] > 1.5],
    ["delta_chi2_true", "chi2_score"],
    ascending=True,
)

selected = pd.concat(selected_rows, ignore_index=True)

# If fewer than 6, fill with best remaining examples
if len(selected) < 6:
    used = set(selected["global_i"].astype(int))
    fill = selected_pool[
        ~selected_pool["global_i"].astype(int).isin(used)
    ].copy()

    fill = fill.sort_values(
        ["delta_chi2_true", "chi2_score", "n_peak_total"],
        ascending=[True, True, False],
    )

    fill = fill.head(6 - len(selected)).copy()
    fill["selection_reason"] = "additional_good_confusion"

    selected = pd.concat([selected, fill], ignore_index=True)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

out = (
    run_dir
    / "figures"
    / "confused_fspl_noparallax"
    / "representative_confused_for_lightcurves.csv"
)

cols = [
    "selection_reason",
    "global_i",
    "chi2_red",
    "delta_chi2_true",
    "n_peak_total",
    "rho_true",
    "rho_fit",
    "rho_ratio",
    "rho_rel_error",
    "sigma_rho_over_rho",
    "tE_true",
    "tE_fit",
    "tE_ratio",
    "u0_true",
    "u0_fit",
    "piE_true",
    "fit_file",
    "results_dir",
]

cols = [c for c in cols if c in selected.columns]

selected = selected[cols].copy()
selected.to_csv(out, index=False)

print("\nSaved selected events:")
print(out)

print("\nSelected events:")
print(selected)

In [ ]:
import pandas as pd
from pathlib import Path

run_dir = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/"
    "LSSTMONTS_xi_baseline_v5p3p5_hiddenParallax_FSPLparallax_fitFSPLNoPiE_t0pm60_DetectionFlag"
)

file = run_dir / "figures/confused_delta_chi2_lt12/representative_delta_chi2_lt12_events.csv"

df = pd.read_csv(file)

cols = [
    "selection_reason",
    "global_i",
    "delta_chi2_true",
    "chi2_red",
    "n_peak_total",
    "rho_true",
    "rho_fit",
    "rho_rel_error",
    "tE_true",
    "tE_fit",
    "tE_ratio",
    "piE_true",
]

cols = [c for c in cols if c in df.columns]

print(df.loc[df["global_i"] == 448, cols].T)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Statistical plots of sigma_rho/rho as a function of injected rho_true.

This script reads the fitted events from a run directory and creates:

    1. sigma_rho/rho_fit vs rho_true
    2. sigma_rho/rho_true vs rho_true
    3. binned median sigma_rho/rho_fit vs rho_true
    4. fraction with sigma_rho/rho_fit < threshold vs rho_true
    5. rho_fit vs rho_true colored by sigma_rho/rho_fit
    6. CSV tables with event-level and binned statistics

Default input:
    runs/<run_name>/

Default output:
    runs/<run_name>/figures/sigma_rho_vs_rho_true_stats/
"""

from pathlib import Path
import argparse
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


DEFAULT_RUN_DIR = Path(
    "/home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/"
    "LSSTMONTS_xi_baseline_v5p3p5_hiddenParallax_FSPLparallax_fitFSPLNoPiE_t0pm60_DetectionFlag"
)


# ============================================================
# I/O helpers
# ============================================================

def find_summary_files(run_dir):
    run_dir = Path(run_dir)

    files = []

    direct = run_dir / "logs" / "run_summary.parquet"
    if direct.exists():
        files.append(direct)

    files.extend(sorted(run_dir.glob("chunk_*/logs/run_summary.parquet")))

    if len(files) == 0:
        raise FileNotFoundError(
            f"No encontré run_summary.parquet en {run_dir}/logs "
            "ni en chunk_*/logs."
        )

    return files


def load_summary(run_dir):
    files = find_summary_files(run_dir)

    tables = []

    for file in files:
        df = pd.read_parquet(file)
        df["summary_file"] = str(file)
        tables.append(df)

    summary = pd.concat(tables, ignore_index=True)

    if "global_i" in summary.columns:
        summary = (
            summary.sort_values("global_i")
            .drop_duplicates("global_i", keep="last")
            .reset_index(drop=True)
        )

    return summary


def trailing_integer_from_path(path):
    """
    Extract trailing integer from filenames like:
        fit_rr_manual_20260941.parquet
        true_rr_manual_20260941.parquet
    """
    stem = Path(path).stem

    match = re.search(r"(\d+)$", stem)

    if match is None:
        return None

    return int(match.group(1))


def build_parquet_index(run_dir):
    """
    Index true_rr and fit_rr parquet files by simulation seed.
    """
    run_dir = Path(run_dir)

    fit_files = sorted(run_dir.rglob("fit_rr_*.parquet"))
    true_files = sorted(run_dir.rglob("true_rr_*.parquet"))

    fit_by_seed = {}
    true_by_seed = {}

    for file in fit_files:
        seed = trailing_integer_from_path(file)
        if seed is not None:
            fit_by_seed[int(seed)] = file

    for file in true_files:
        seed = trailing_integer_from_path(file)
        if seed is not None:
            true_by_seed[int(seed)] = file

    return fit_by_seed, true_by_seed


def read_first_row(file):
    if file is None:
        return {}

    try:
        df = pd.read_parquet(file)
    except Exception:
        return {}

    if len(df) == 0:
        return {}

    return df.iloc[0].to_dict()


def safe_float(value, default=np.nan):
    try:
        if value is None:
            return default

        value = np.asarray(value)

        if value.size == 0:
            return default

        if value.size == 1:
            return float(value.ravel()[0])

        return default

    except Exception:
        return default


def get_first(row, names, default=np.nan):
    for name in names:
        if name in row:
            value = safe_float(row[name], default=np.nan)
            if np.isfinite(value):
                return value

    return default


def extract_sigma_rho(fit_row):
    """
    Prefer rho_err from the fit_rr parquet.

    If rho_err is not present, try covariance_matrix as fallback.
    """
    sigma = get_first(
        fit_row,
        [
            "rho_err",
            "rho_err_cov",
            "sigma_rho",
            "sigma_rho_fit",
        ],
        default=np.nan,
    )

    if np.isfinite(sigma):
        return sigma

    if "covariance_matrix" not in fit_row:
        return np.nan

    try:
        cov = np.asarray(fit_row["covariance_matrix"], dtype=float)

        if cov.ndim != 2:
            return np.nan

        # For FSPL without parallax the usual geometric order is:
        # t0, u0, tE, rho, ...
        rho_index = 3

        if cov.shape[0] <= rho_index or cov.shape[1] <= rho_index:
            return np.nan

        var = float(cov[rho_index, rho_index])

        if np.isfinite(var) and var >= 0:
            return float(np.sqrt(var))

    except Exception:
        return np.nan

    return np.nan


# ============================================================
# Build event metrics
# ============================================================

def build_metrics_from_run(run_dir, max_events=None):
    run_dir = Path(run_dir)

    summary = load_summary(run_dir)

    if "status" in summary.columns:
        summary = summary[summary["status"].astype(str).isin(["ok", "fitted"])].copy()

    if max_events is not None:
        summary = summary.head(int(max_events)).copy()

    fit_by_seed, true_by_seed = build_parquet_index(run_dir)

    rows = []

    print("=" * 80)
    print("Reading event parquets")
    print("=" * 80)
    print("N summary      =", len(summary))
    print("N fit_rr files =", len(fit_by_seed))
    print("N true files   =", len(true_by_seed))
    print("=" * 80)

    for k, (_, srow) in enumerate(summary.iterrows(), start=1):
        seed = int(srow["simulation_seed"])

        fit_file = fit_by_seed.get(seed, None)
        true_file = true_by_seed.get(seed, None)

        fit_row = read_first_row(fit_file)
        true_row = read_first_row(true_file)

        global_i = get_first(srow, ["global_i"], default=np.nan)

        rho_true = get_first(
            true_row,
            ["rho", "rho_true", "rho_catalog", "true_rho"],
            default=np.nan,
        )

        if not np.isfinite(rho_true):
            rho_true = get_first(
                srow,
                ["true_rho", "rho_catalog", "rho_true", "rho"],
                default=np.nan,
            )

        rho_fit = get_first(
            fit_row,
            ["rho", "rho_fit", "fit_rho"],
            default=np.nan,
        )

        sigma_rho = extract_sigma_rho(fit_row)

        chi2_fit = get_first(
            fit_row,
            ["chichi", "chi2", "chi2_fit"],
            default=np.nan,
        )

        dof = get_first(
            fit_row,
            ["dof", "dof_fit"],
            default=np.nan,
        )

        chi2_true = get_first(
            fit_row,
            ["chi2_true"],
            default=np.nan,
        )

        if not np.isfinite(chi2_true):
            chi2_true = get_first(
                true_row,
                ["chi2_true"],
                default=np.nan,
            )

        delta_chi2 = get_first(
            fit_row,
            ["delta_chi2_true", "delta_chi2", "Delta_chi2"],
            default=np.nan,
        )

        if not np.isfinite(delta_chi2):
            if np.isfinite(chi2_fit) and np.isfinite(chi2_true):
                delta_chi2 = chi2_fit - chi2_true

        chi2_red = np.nan
        if np.isfinite(chi2_fit) and np.isfinite(dof) and dof > 0:
            chi2_red = chi2_fit / dof

        tE_true = get_first(true_row, ["tE", "tE_true"], default=np.nan)
        tE_fit = get_first(fit_row, ["tE", "tE_fit"], default=np.nan)

        u0_true = get_first(true_row, ["u0", "u0_true"], default=np.nan)
        u0_fit = get_first(fit_row, ["u0", "u0_fit"], default=np.nan)

        piE_true = get_first(true_row, ["piE", "piE_true"], default=np.nan)

        if not np.isfinite(piE_true):
            piEN = get_first(true_row, ["piEN", "piEN_true"], default=np.nan)
            piEE = get_first(true_row, ["piEE", "piEE_true"], default=np.nan)

            if np.isfinite(piEN) and np.isfinite(piEE):
                piE_true = float(np.hypot(piEN, piEE))

        n_peak_cols = [
            c for c in true_row.keys()
            if isinstance(c, str) and c.endswith("_peak")
        ]

        if len(n_peak_cols) > 0:
            n_peak_total = np.nansum([
                safe_float(true_row.get(c, 0.0), default=0.0)
                for c in n_peak_cols
            ])
        else:
            n_peak_total = np.nan

        sigma_over_rho_fit = np.nan
        if np.isfinite(sigma_rho) and np.isfinite(rho_fit) and rho_fit > 0:
            sigma_over_rho_fit = sigma_rho / rho_fit

        sigma_over_rho_true = np.nan
        if np.isfinite(sigma_rho) and np.isfinite(rho_true) and rho_true > 0:
            sigma_over_rho_true = sigma_rho / rho_true

        rho_rel_error = np.nan
        rho_ratio = np.nan

        if np.isfinite(rho_true) and rho_true > 0 and np.isfinite(rho_fit):
            rho_rel_error = (rho_fit - rho_true) / rho_true
            rho_ratio = rho_fit / rho_true

        at_u0_bound = False
        if np.isfinite(u0_fit):
            at_u0_bound = np.isclose(abs(u0_fit), 5.0, rtol=0, atol=1e-2)

        at_tE_bound = False
        if np.isfinite(tE_fit):
            at_tE_bound = np.isclose(tE_fit, 20000.0, rtol=0, atol=1e-2)

        bad_boundary_fit = bool(at_u0_bound or at_tE_bound)

        rows.append(
            {
                "global_i": global_i,
                "simulation_seed": seed,
                "fit_file": str(fit_file) if fit_file is not None else "",
                "true_file": str(true_file) if true_file is not None else "",
                "rho_true": rho_true,
                "rho_fit": rho_fit,
                "sigma_rho": sigma_rho,
                "sigma_rho_over_rho_fit": sigma_over_rho_fit,
                "sigma_rho_over_rho_true": sigma_over_rho_true,
                "rho_ratio": rho_ratio,
                "rho_rel_error": rho_rel_error,
                "abs_rho_rel_error": abs(rho_rel_error) if np.isfinite(rho_rel_error) else np.nan,
                "tE_true": tE_true,
                "tE_fit": tE_fit,
                "tE_ratio": tE_fit / tE_true if np.isfinite(tE_fit) and np.isfinite(tE_true) and tE_true > 0 else np.nan,
                "u0_true": u0_true,
                "u0_fit": u0_fit,
                "piE_true": piE_true,
                "chi2_fit": chi2_fit,
                "chi2_true": chi2_true,
                "delta_chi2": delta_chi2,
                "dof": dof,
                "chi2_red": chi2_red,
                "n_peak_total": n_peak_total,
                "bad_boundary_fit": bad_boundary_fit,
            }
        )

        if k == 1 or k % 500 == 0 or k == len(summary):
            print(f"{k}/{len(summary)}")

    metrics = pd.DataFrame(rows)

    metrics["rho_measured_fit"] = (
        np.isfinite(metrics["sigma_rho_over_rho_fit"])
        & (metrics["sigma_rho_over_rho_fit"] < 0.5)
    )

    metrics["rho_measured_true"] = (
        np.isfinite(metrics["sigma_rho_over_rho_true"])
        & (metrics["sigma_rho_over_rho_true"] < 0.5)
    )

    metrics["rho_recovered_50pct"] = (
        np.isfinite(metrics["abs_rho_rel_error"])
        & (metrics["abs_rho_rel_error"] < 0.5)
    )

    metrics["not_distinguishable_delta12"] = (
        np.isfinite(metrics["delta_chi2"])
        & (metrics["delta_chi2"] >= 0)
        & (metrics["delta_chi2"] < 12)
    )

    metrics["good_no_parallax_fit"] = (
        np.isfinite(metrics["chi2_red"])
        & (metrics["chi2_red"] < 1.5)
        & (~metrics["bad_boundary_fit"])
    )

    metrics["good_peak"] = (
        np.isfinite(metrics["n_peak_total"])
        & (metrics["n_peak_total"] >= 5)
    )

    metrics["confused_delta12_clean"] = (
        metrics["good_no_parallax_fit"]
        & metrics["good_peak"]
        & metrics["not_distinguishable_delta12"]
    )

    return metrics


# ============================================================
# Binned statistics
# ============================================================

def make_log_bins(x, nbins=12):
    x = np.asarray(x, dtype=float)

    good = np.isfinite(x) & (x > 0)

    if good.sum() == 0:
        raise RuntimeError("No hay valores positivos de rho_true para binnear.")

    xmin = np.nanmin(x[good])
    xmax = np.nanmax(x[good])

    return np.logspace(np.log10(xmin), np.log10(xmax), nbins + 1)


def binned_sigma_statistics(df, nbins=12, measurement_threshold=0.5):
    df = df.copy()

    bins = make_log_bins(df["rho_true"], nbins=nbins)

    df["rho_bin"] = pd.cut(
        df["rho_true"],
        bins=bins,
        include_lowest=True,
    )

    rows = []

    for interval, group in df.groupby("rho_bin", observed=True):
        if len(group) == 0:
            continue

        x_left = float(interval.left)
        x_right = float(interval.right)
        x_center = np.sqrt(x_left * x_right)

        y = group["sigma_rho_over_rho_fit"].to_numpy(float)
        y = y[np.isfinite(y) & (y > 0)]

        measured = group["rho_measured_fit"].to_numpy(bool)

        n = len(group)
        n_finite_sigma = len(y)
        n_measured = int(np.nansum(measured))
        frac_measured = n_measured / n if n > 0 else np.nan

        frac_err = np.sqrt(frac_measured * (1.0 - frac_measured) / n) if n > 0 else np.nan

        if len(y) > 0:
            logy = np.log10(y)

            median = float(np.nanmedian(y))
            q16 = float(10.0 ** np.nanpercentile(logy, 16))
            q84 = float(10.0 ** np.nanpercentile(logy, 84))
        else:
            median = np.nan
            q16 = np.nan
            q84 = np.nan

        rows.append(
            {
                "rho_bin_left": x_left,
                "rho_bin_right": x_right,
                "rho_bin_center": x_center,
                "n": n,
                "n_finite_sigma": n_finite_sigma,
                "n_measured_sigma_rho_over_rho_fit_lt_threshold": n_measured,
                "measurement_threshold": measurement_threshold,
                "frac_measured": frac_measured,
                "frac_measured_err_binomial": frac_err,
                "median_sigma_rho_over_rho_fit": median,
                "q16_sigma_rho_over_rho_fit": q16,
                "q84_sigma_rho_over_rho_fit": q84,
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# Plots
# ============================================================

def savefig(fig, outdir, name):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    fig.savefig(outdir / f"{name}.png", dpi=250, bbox_inches="tight")
    fig.savefig(outdir / f"{name}.pdf", bbox_inches="tight")

    plt.close(fig)


def plot_sigma_vs_rho_true(df, outdir, ycol, name, ylabel):
    good = (
        np.isfinite(df["rho_true"])
        & np.isfinite(df[ycol])
        & (df["rho_true"] > 0)
        & (df[ycol] > 0)
    )

    sub = df[good].copy()

    selected = sub[sub["confused_delta12_clean"]].copy()

    fig, ax = plt.subplots(figsize=(7.2, 5.6))

    ax.scatter(
        sub["rho_true"],
        sub[ycol],
        s=10,
        alpha=0.18,
        label="All fitted events",
    )

    if len(selected) > 0:
        ax.scatter(
            selected["rho_true"],
            selected[ycol],
            s=16,
            alpha=0.75,
            label=r"Clean confused: $0\leq\Delta\chi^2<12$",
        )

    ax.axhline(
        0.5,
        linestyle="--",
        linewidth=1.2,
        label=r"$\sigma_\rho/\rho=0.5$",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"Injected $\rho_{\rm true}$")
    ax.set_ylabel(ylabel)

    ax.set_title(r"Finite-source precision as a function of injected $\rho$")

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    savefig(fig, outdir, name)


def plot_binned_sigma(binned, outdir):
    good = (
        np.isfinite(binned["rho_bin_center"])
        & np.isfinite(binned["median_sigma_rho_over_rho_fit"])
    )

    b = binned[good].copy()

    fig, ax = plt.subplots(figsize=(7.2, 5.4))

    yerr_low = (
        b["median_sigma_rho_over_rho_fit"]
        - b["q16_sigma_rho_over_rho_fit"]
    )

    yerr_high = (
        b["q84_sigma_rho_over_rho_fit"]
        - b["median_sigma_rho_over_rho_fit"]
    )

    ax.errorbar(
        b["rho_bin_center"],
        b["median_sigma_rho_over_rho_fit"],
        yerr=[yerr_low, yerr_high],
        fmt="o-",
        capsize=3,
        label="Median and 16--84 percentile range",
    )

    ax.axhline(
        0.5,
        linestyle="--",
        linewidth=1.2,
        label=r"$\sigma_\rho/\rho_{\rm fit}=0.5$",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"Injected $\rho_{\rm true}$")
    ax.set_ylabel(r"$\sigma_\rho/\rho_{\rm fit}$")
    ax.set_title(r"Binned finite-source precision")

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    savefig(fig, outdir, "binned_sigma_rho_over_rho_fit_vs_rho_true")


def plot_fraction_measured(binned, outdir):
    good = (
        np.isfinite(binned["rho_bin_center"])
        & np.isfinite(binned["frac_measured"])
    )

    b = binned[good].copy()

    fig, ax = plt.subplots(figsize=(7.2, 5.4))

    ax.errorbar(
        b["rho_bin_center"],
        b["frac_measured"],
        yerr=b["frac_measured_err_binomial"],
        fmt="o-",
        capsize=3,
    )

    ax.set_xscale("log")
    ax.set_ylim(-0.05, 1.05)

    ax.set_xlabel(r"Injected $\rho_{\rm true}$")
    ax.set_ylabel(r"Fraction with $\sigma_\rho/\rho_{\rm fit}<0.5$")
    ax.set_title(r"Finite-source measurement fraction")

    for _, row in b.iterrows():
        ax.text(
            row["rho_bin_center"],
            row["frac_measured"] + 0.035,
            f"N={int(row['n'])}",
            ha="center",
            va="bottom",
            fontsize=7,
            rotation=45,
        )

    ax.grid(True, alpha=0.3)

    savefig(fig, outdir, "fraction_sigma_rho_over_rho_fit_lt_0p5_vs_rho_true")


def plot_rho_fit_vs_true_colored_sigma(df, outdir):
    good = (
        np.isfinite(df["rho_true"])
        & np.isfinite(df["rho_fit"])
        & np.isfinite(df["sigma_rho_over_rho_fit"])
        & (df["rho_true"] > 0)
        & (df["rho_fit"] > 0)
        & (df["sigma_rho_over_rho_fit"] > 0)
    )

    sub = df[good].copy()

    if len(sub) == 0:
        print("[warning] No hay datos para rho_fit vs rho_true colored sigma.")
        return

    c = np.log10(sub["sigma_rho_over_rho_fit"])

    fig, ax = plt.subplots(figsize=(6.8, 5.8))

    sc = ax.scatter(
        sub["rho_true"],
        sub["rho_fit"],
        c=c,
        s=12,
        alpha=0.75,
    )

    lo = np.nanmin([sub["rho_true"].min(), sub["rho_fit"].min()])
    hi = np.nanmax([sub["rho_true"].max(), sub["rho_fit"].max()])

    ax.plot(
        [lo, hi],
        [lo, hi],
        linestyle="--",
        linewidth=1.2,
        label=r"$\rho_{\rm fit}=\rho_{\rm true}$",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"Injected $\rho_{\rm true}$")
    ax.set_ylabel(r"Recovered $\rho_{\rm fit}$")
    ax.set_title(r"$\rho$ recovery colored by $\log_{10}(\sigma_\rho/\rho_{\rm fit})$")

    cb = fig.colorbar(sc, ax=ax)
    cb.set_label(r"$\log_{10}(\sigma_\rho/\rho_{\rm fit})$")

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    savefig(fig, outdir, "rho_fit_vs_rho_true_colored_by_sigma_rho_over_rho_fit")


def plot_sigma_histograms(df, outdir):
    good = (
        np.isfinite(df["sigma_rho_over_rho_fit"])
        & (df["sigma_rho_over_rho_fit"] > 0)
    )

    sub = df[good].copy()

    selected = sub[sub["confused_delta12_clean"]].copy()

    fig, ax = plt.subplots(figsize=(7.0, 5.2))

    x = np.log10(sub["sigma_rho_over_rho_fit"])

    lo = np.nanpercentile(x, 1)
    hi = np.nanpercentile(x, 99)

    bins = np.linspace(lo, hi, 60)

    ax.hist(
        x,
        bins=bins,
        histtype="step",
        linewidth=1.5,
        label="All fitted events",
    )

    if len(selected) > 0:
        xs = np.log10(selected["sigma_rho_over_rho_fit"])
        ax.hist(
            xs,
            bins=bins,
            histtype="stepfilled",
            alpha=0.35,
            label=r"Clean confused: $0\leq\Delta\chi^2<12$",
        )

    ax.axvline(
        np.log10(0.5),
        linestyle="--",
        linewidth=1.2,
        label=r"$\sigma_\rho/\rho_{\rm fit}=0.5$",
    )

    ax.set_xlabel(r"$\log_{10}(\sigma_\rho/\rho_{\rm fit})$")
    ax.set_ylabel("Number of events")
    ax.set_title(r"Distribution of finite-source precision")

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    savefig(fig, outdir, "hist_log_sigma_rho_over_rho_fit")


def make_all_plots(metrics, binned, outdir):
    plot_sigma_vs_rho_true(
        metrics,
        outdir,
        ycol="sigma_rho_over_rho_fit",
        name="sigma_rho_over_rho_fit_vs_rho_true",
        ylabel=r"$\sigma_\rho/\rho_{\rm fit}$",
    )

    plot_sigma_vs_rho_true(
        metrics,
        outdir,
        ycol="sigma_rho_over_rho_true",
        name="sigma_rho_over_rho_true_vs_rho_true",
        ylabel=r"$\sigma_\rho/\rho_{\rm true}$",
    )

    plot_binned_sigma(binned, outdir)
    plot_fraction_measured(binned, outdir)
    plot_rho_fit_vs_true_colored_sigma(metrics, outdir)
    plot_sigma_histograms(metrics, outdir)


# ============================================================
# Main
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--run-dir",
        default=str(DEFAULT_RUN_DIR),
    )

    parser.add_argument(
        "--output-subdir",
        default="sigma_rho_vs_rho_true_stats",
    )

    parser.add_argument(
        "--nbins",
        type=int,
        default=12,
    )

    parser.add_argument(
        "--measurement-threshold",
        type=float,
        default=0.5,
    )

    parser.add_argument(
        "--max-events",
        type=int,
        default=None,
    )

    args = parser.parse_args()

    run_dir = Path(args.run_dir)

    outdir = run_dir / "figures" / args.output_subdir
    outdir.mkdir(parents=True, exist_ok=True)

    metrics = build_metrics_from_run(
        run_dir=run_dir,
        max_events=args.max_events,
    )

    metrics_file = outdir / "event_metrics_sigma_rho_vs_rho_true.csv"
    metrics.to_csv(metrics_file, index=False)

    valid_for_bins = metrics[
        np.isfinite(metrics["rho_true"])
        & (metrics["rho_true"] > 0)
        & np.isfinite(metrics["sigma_rho_over_rho_fit"])
        & (metrics["sigma_rho_over_rho_fit"] > 0)
    ].copy()

    binned = binned_sigma_statistics(
        valid_for_bins,
        nbins=args.nbins,
        measurement_threshold=args.measurement_threshold,
    )

    binned_file = outdir / "binned_sigma_rho_vs_rho_true_stats.csv"
    binned.to_csv(binned_file, index=False)

    make_all_plots(
        metrics=metrics,
        binned=binned,
        outdir=outdir,
    )

    print("=" * 80)
    print("DONE")
    print("=" * 80)
    print("Output directory:")
    print(outdir)
    print()
    print("Tables:")
    print(metrics_file)
    print(binned_file)
    print()
    print("Main figures:")
    for file in sorted(outdir.glob("*.png")):
        print(file)
    print("=" * 80)

    print("\nSummary")
    print("-------")
    print("N total metrics:", len(metrics))
    print("finite rho_true:", np.isfinite(metrics["rho_true"]).sum())
    print("finite rho_fit:", np.isfinite(metrics["rho_fit"]).sum())
    print("finite sigma_rho:", np.isfinite(metrics["sigma_rho"]).sum())
    print(
        f"N sigma_rho/rho_fit < {args.measurement_threshold}:",
        int(metrics["rho_measured_fit"].sum()),
    )
    print(
        f"fraction sigma_rho/rho_fit < {args.measurement_threshold}:",
        float(metrics["rho_measured_fit"].mean()),
    )
    print(
        "N clean confused delta12:",
        int(metrics["confused_delta12_clean"].sum()),
    )


if __name__ == "__main__":
    main()